# Avance 2. Ingeniería de características para imágenes DESI

**Proyecto:** Inferencia de propiedades físicas de galaxias a partir de imágenes multibanda.
**Instituto Tecnológico y de Estudios Superiores de Monterrey** · Maestría en Inteligencia Artificial Aplicada · Equipo 24.
**Colaboración científica:** Instituto de Astronomía, UNAM.

José Florencio Maguey Peralta — A01796727
Erick Alberto Bustos Cruz — A01378966
Luz Copelia Minutti Pérez — A01796921

**Asesores:** Dr. Iván Reyes Amezcua; Dr. Héctor Hernández Toledo; Dr. José Antonio Vázquez Mata.

---

**Fase de CRISP-ML(Q):** 2, *Data Engineering (Data Preparation)* (Studer et al. 2021).
**Objetivos del avance:** 2.3, crear nuevas características que mejoren el rendimiento de los modelos; 2.4, mitigar el riesgo de características sesgadas y acelerar la convergencia de algunos algoritmos.

---

**Quiénes somos.** Somos estudiantes de la Maestría en Inteligencia Artificial Aplicada. Venimos de la computación y la ciencia de datos, no de la astronomía: lo que sabemos de galaxias lo aprendimos durante el proyecto, con los asesores del Instituto de Astronomía y con los artículos que citamos. Por eso, en lo astronómico usamos definiciones publicadas en lugar de inventar las nuestras. Cuando una decisión es nuestra y no de la literatura, lo decimos y la medimos.

**Cómo está armado este cuaderno.** Reúne el trabajo de los tres integrantes. Cada aporte está en la sección del flujo donde se usa, adaptado a los mismos datos (cubos g, r, z de DESI DR10) y con una prueba que justifica la decisión:

| Cuaderno de origen | Autor | Qué aporta | Dónde está |
|---|---|---|---|
| `Avance2.24` (ingeniería de características tabulares) | Erick Alberto Bustos Cruz | Medición de 49 características físicas, codificación, transformación, escalamiento, sesgos, selección, extracción y comparación con la red | Secciones 4 y 6–14 |
| `Preprocesamiento_v1.3.1` (preparación de la imagen) | José Florencio Maguey Peralta | Fondo del cielo con `Background2D`, colapso de bandas, comparación de 27 normalizaciones con entropía y saturación, máscara gaussiana con estrellas, contorno activo y canal de grumosidad | Secciones 5.2, 5.3 y 6.7 |
| `DESI_Entrenamiento_Mejorado` (protocolo de la siguiente fase) | Luz Copelia Minutti Pérez | Auditoría de las entradas, representaciones de las bandas para la red, estandarización de etiquetas y redshift, aumentos sin interpolación y diseño de los experimentos de modelado | Secciones 4.1, 5.4–5.6 y 15 |

Las funciones que vienen de los cuadernos del equipo están en `src/caracteristicas_desi/imagen_equipo.py`; en su documentación se indica de dónde sale cada una y qué cambiamos para que funcione en las 9,303 galaxias.


## 1. Qué hacemos y por qué

Como punto de partida tenemos una red convolucional (la cadena de Katachi; Alfonzo et al. 2024) que entrenamos con las imágenes DESI y que estima la masa estelar, la tasa de formación estelar (SFR) y el índice D4000 directamente a partir de los píxeles. En las galaxias de validación obtuvo un RMSE de **0.224 dex** en masa, **0.631 dex** en SFR y **0.075** en D4000.

Esa red recibe 270,000 valores por galaxia (3 bandas × 300 × 300 píxeles) y no incorpora conocimiento físico. En este avance tomamos el camino contrario: reducimos cada imagen a **unas decenas de variables con significado físico**, medidas con los métodos habituales de la astronomía extragaláctica, y las preparamos para el aprendizaje automático. Antes de eso documentamos la preparación de la imagen que recibe la red, que también es ingeniería de características, a nivel de píxel:

0. **Preparación de la imagen para la red** (sección 5): fondo del cielo, máscaras, estiramiento, representaciones de las bandas y escala de los planos y las etiquetas.
1. **Construcción** (sección 6): fotometría, colores, gradientes de color, tamaños, concentración, asimetría, distribución de la luz y variables que dependen de la distancia.
2. **Codificación y discretización** (sección 8).
3. **Transformación y escalamiento** (secciones 9 y 10), con una prueba de convergencia.
4. **Sesgos** (sección 11): diferencias entre telescopios y con la distancia.
5. **Selección y extracción** (secciones 12 y 13): filtros, PCA, análisis factorial y eigengalaxias.
6. **Comparación** (sección 14): cuánto predicen las características frente a la red convolucional.

La pregunta que buscamos responder es:

> **¿Cuánto de la masa estelar, la SFR y el D4000 se puede predecir con unas decenas de características bien construidas, comparado con una red que usa la imagen completa?**

**Criterios generales.** Todos los ajustes (transformaciones, escaladores, selección, PCA y análisis factorial) se hacen **solo con las galaxias de entrenamiento** y se aplican sin cambios a las de validación. Las **galaxias de prueba no se usan**: las reservamos para la evaluación final del proyecto.

### 1.1 Antecedentes: lo que aprendimos al entrenar la red

Antes de este avance reprodujimos Katachi con los FITS de DESI. Ese trabajo definió la muestra, preparó las imágenes y dejó varios resultados que orientan las decisiones de este cuaderno. Todos los valores son de las galaxias de validación.

| Corrida | log M\* [dex] | log SFR [dex] | D4000 |
|---|---|---|---|
| Red con capas congeladas, como indica el artículo | 0.296 | 0.826 | 0.103 |
| Red con todas las capas entrenables | **0.224** | **0.631** | **0.075** |
| Katachi publicado (sus galaxias de prueba, como referencia) | 0.222 | 0.668 | 0.100 |

| Hallazgo en la red | Qué decisión motiva en este cuaderno |
|---|---|
| La masa se sobreestima en galaxias cercanas (+0.12 dex a z < 0.025) y se subestima en las lejanas (hasta −0.10 dex) | Construir variables intrínsecas con el redshift (sección 6.4) y revisar el sesgo por distancia (secciones 11 y 14.1) |
| La SFR casi nunca baja de log SFR ≈ −2.5: la red no distingue entre galaxias que ya casi no forman estrellas | Clases formadora/apagada para la selección (sección 8.2) y variables de color y gradiente de color |
| El 45 % de las galaxias tiene imágenes de BASS/MzLS y el 55 % de DECam | Variable `region_norte` y revisión de sesgos por telescopio (sección 11) |
| Con capas congeladas la red era seis veces más sensible al ruido; sin congelar, su variación ante el ruido real del cielo (0.025 dex en masa) es como la de Katachi (0.023) | Revisión de la preparación de la imagen a nivel de píxel (sección 5) |
| En las primeras épocas la pérdida de la SFR tenía picos de hasta 10 | Revisión de la escala de los planos de masa y SFR que recibe la red (sección 5.5) |

**Qué viene de ese trabajo y qué es nuevo en este avance:**

| Elemento | Procedencia |
|---|---|
| Galaxias, etiquetas de Pipe3D, cortes de masa y SFR del artículo, partición y corte de sSFR = −10.8 | Katachi (Alfonzo et al. 2024) |
| Recortes de 300 × 300 px y preparación de la imagen para la red (sección 5) | Nuestra reproducción de Katachi con FITS de DESI |
| Resultados de la red usados como referencia | Nuestra reproducción de Katachi |
| Máscara de la galaxia | Evaluación de máscaras del equipo |
| Fondo del cielo, normalizaciones y canal de grumosidad (secciones 5.2, 5.3 y 6.7) | Cuaderno de Preprocesamiento del equipo |
| Auditoría de las entradas, representaciones y estandarización para la red (secciones 4.1 y 5.4–5.6) | Cuaderno de Entrenamiento del equipo |
| Características tabulares, codificación, transformaciones, escalamiento, sesgos, selección y extracción (secciones 6 a 14) | Nuevo en este avance |

### 1.2 Dónde se cubre cada criterio de la rúbrica

| Criterio | Qué hacemos | Secciones |
|---|---|---|
| **Construcción** | Características físicas con definiciones publicadas y validadas contra otros catálogos; canales de grumosidad; variables intrínsecas con el redshift; codificación one-hot y ordinal | 5.2, 6, 6.7, 8 |
| **Normalización** | Imagen: fondo del cielo, comparación de estiramientos, representaciones para la red y estandarización de etiquetas y redshift. Tabla: log, Box-Cox y Yeo-Johnson con gráficas Q-Q, comparación de escaladores y prueba de convergencia | 5.3–5.5, 9, 10 |
| **Selección y extracción** | Umbral de varianza, correlación, F, información mutua, χ² y ANOVA, con el número de variables elegido por validación cruzada; PCA, análisis factorial, eigengalaxias y Zoobot | 12, 13, 14 |
| **Conclusiones** | Preparación de los datos en el marco de CRISP-ML(Q) y paso a la fase de modelado | 15 |

### Cómo ejecutar

1. Abrir el cuaderno en Colab. Funciona con CPU; si hay GPU, también se ejecuta la sección opcional 13.4 (Zoobot).
2. Ejecutar todas las celdas en orden (*Entorno de ejecución → Ejecutar todo*). La primera ejecución tarda entre 40 y 60 minutos con un entorno de muchos procesadores (por ejemplo, el de GPU A100), principalmente por la medición de las 9,303 galaxias y el recorrido de la sección 4.1. Si Colab se desconecta, basta con ejecutar todo de nuevo: las mediciones continúan desde el último bloque guardado.
3. El cuaderno **solo lee** el catálogo, los FITS y la caché de imágenes. Los resultados se guardan en `Mi unidad/resultados_fe/`.

## 2. Parámetros

Las constantes de la medición (umbrales de segmentación, definición del radio de Petrosian, coeficientes de extinción y cosmología) están en `src/caracteristicas_desi/config.py`, cada una con su fuente. En esta sección solo definimos los parámetros de ejecución.

| Parámetro | Valor | Significado |
|---|---|---|
| `SEMILLA` | 42 | Semilla de todos los procesos aleatorios |
| `N_BOOT` | 1000 | Remuestreos *bootstrap* para los intervalos de confianza del RMSE |
| `UMBRAL_ASIMETRIA` | 1.0 | Una variable se transforma si su asimetría en entrenamiento supera este valor en valor absoluto (sección 9) |
| `USAR_ZOOBOT` | `"auto"` | La extracción con Zoobot (13.4) solo corre si hay GPU |
| `N_DEMO_FITS` | 6 | Cubos completos de 800 px con los que se comparan el fondo del cielo y las máscaras (5.2) |

In [ ]:
import os
env = os.environ.get          # las variables AV2_* solo se usan en la prueba local reducida

SEMILLA = 42
N_BOOT = int(env("AV2_N_BOOT", "1000"))
UMBRAL_ASIMETRIA = 1.0
USAR_ZOOBOT = env("AV2_ZOOBOT", "auto")      # "auto", "si" o "no"
N_DEMO_FITS = 6
OBJETIVOS = ["log_mstar", "log_sfr", "d4000"]
NOMBRES = {"log_mstar": "log M* [dex]", "log_sfr": "log SFR [dex]", "d4000": "D4000"}
# Referencias de la red convolucional (validación) y de Katachi publicado (prueba), RMSE
CNN = {"log_mstar": 0.224, "log_sfr": 0.631, "d4000": 0.075}
KATACHI = {"log_mstar": 0.222, "log_sfr": 0.668, "d4000": 0.100}

## 3. Entorno

Descargamos el código del repositorio del equipo, de modo que cada resultado quede ligado a una versión concreta, e instalamos las dependencias que Colab no incluye: `photutils`, `dustmaps` y `factor_analyzer` (y `scikit-image`, por si el entorno no la trae).

In [ ]:
import sys, json, time, subprocess, platform, warnings
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/erteck/InferenciaPropiedadesFisicas.git"
REPO_DIR = Path(env("AV2_REPO_DIR", "/content/InferenciaPropiedadesFisicas"))

if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "photutils", "dustmaps", "factor_analyzer", "scikit-image"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)

sys.path.insert(0, str(REPO_DIR))
COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Python", platform.python_version(), "· commit del código:", COMMIT[:12])

In [ ]:
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.table import Table
from scipy import ndimage, stats
from astropy.stats import sigma_clipped_stats
from astropy.visualization import make_lupton_rgb
from IPython.display import display

from src.caracteristicas_desi import config as FE
from src.caracteristicas_desi import imagen_equipo as IE
from src.caracteristicas_desi.diccionario import DICCIONARIO, grupo
from src.caracteristicas_desi.fisica import agregar_variables_fisicas, ebv_sfd
from src.caracteristicas_desi.fotometria import medir_fotometria
from src.caracteristicas_desi.pipeline import medir_cache
from src.caracteristicas_desi.transformaciones import TransformadorAsimetria
from src.katachi_fits import config as KC
from src.katachi_fits.io_guard import WriteGuard, sha256

pd.set_option("display.precision", 3, "display.max_columns", 40, "display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})
rng = np.random.default_rng(SEMILLA)

def rmse(y, p):
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(p)) ** 2)))

def rmse_ic(y, p, n=N_BOOT, seed=0):
    # Intervalo de confianza del 95 % del RMSE por bootstrap de galaxias.
    y, p = np.asarray(y), np.asarray(p)
    i = np.random.default_rng(seed).integers(0, len(y), (n, len(y)))
    return np.percentile(np.sqrt(((y[i] - p[i]) ** 2).mean(1)), [2.5, 97.5])

### 3.1 Dónde están los datos

| Entrada (solo lectura) | Ubicación en Drive |
|---|---|
| Catálogo maestro del equipo (etiquetas, calidad, particiones) | `descarga_22fb149046f162f3/catalogo_candidato_9661_galaxias.csv` |
| Caché de recortes de 300 × 300 px (78.6″) de los FITS | `resultados_katachi_fits/cache/desi/recorte300/` |
| Catálogo auxiliar (morfología y masas de otros catálogos) | `Datos/inferencia.fits`, en el repositorio |
| FITS completos de 800 × 800 px (solo para la sección 5.2) | `descarga_22fb149046f162f3/fits/` |

La caché se generó con el cuaderno de Katachi a partir de los FITS. Si no existe, la siguiente celda la construye con el mismo código (unos 15 minutos). El código no permite escribir fuera de `resultados_fe/`.

In [ ]:
BASE = Path(env("AV2_BASE", "/content/drive/MyDrive"))     # "Mi unidad" de Google Drive
DESCARGA = BASE / KC.DOWNLOAD_DIRNAME
CACHE = BASE / KC.RESULTS_DIRNAME / "cache" / "desi" / "recorte300"
RESULTADOS = BASE / "resultados_fe"

NOMBRES_CATALOGO = ("catalogo_candidato_9661_galaxias.csv", "catalogo_candidatas_original.csv")
CATALOGO = next((p for n in NOMBRES_CATALOGO for p in DESCARGA.glob(f"**/{n}")), None)
assert CATALOGO is not None, f"No se encontró el catálogo del equipo en {DESCARGA}"
guard = WriteGuard(RESULTADOS, [DESCARGA, CACHE])
guard.mkdir(RESULTADOS)

SHA_CAT = sha256(CATALOGO)
print("Catálogo:", CATALOGO.name, "· ¿versión acordada?", SHA_CAT == KC.TEAM_CATALOG_SHA256)
assert SHA_CAT == KC.TEAM_CATALOG_SHA256 or env("AV2_BASE"), "El catálogo no es la versión acordada"

if not any(CACHE.glob("chunk_*.npz")):
    print("No hay caché de recortes: se construye a partir de los FITS.")
    from src.katachi_fits import catalog as KCat
    from src.katachi_fits.cache import build_cache
    from src.katachi_fits.config import FitsConfig
    plan = KCat.load_plan(DESCARGA / "plan.csv", DESCARGA / "fits")
    equipo = pd.read_csv(CATALOGO, dtype={"mangaid": str})
    plan = plan[plan.mangaid.isin(equipo.loc[equipo.QCFLAG_original == 0, "mangaid"])]
    guard_k = WriteGuard(BASE / KC.RESULTS_DIRNAME, [DESCARGA])
    build_cache(plan, "ruta_local", CACHE, guard_k, FitsConfig(), chunk=500, workers=24)
print("Caché:", CACHE, "·", len(list(CACHE.glob("chunk_*.npz"))), "bloques")
print("Resultados:", RESULTADOS)

## 4. Datos y selección de la muestra

**Selección.** Usamos las galaxias con `QCFLAG = 0`, es decir, aquellas cuyo ajuste espectral en Pipe3D (Sánchez et al. 2022) pasó todos los controles de calidad. Es el criterio que acordamos con los asesores. Conservamos la partición del proyecto (`split_proyecto`).

**Etiquetas.** log M\*, log SFR y D4000 de Pipe3D, **sin transformar**: log M\* y log SFR ya son logaritmos, y D4000 tiene una distribución bimodal (galaxias jóvenes y viejas) que ninguna transformación monótona vuelve normal. Además, de esta forma los resultados quedan en las mismas unidades que los de la red y Katachi.

**Variables que no entran al modelo.** Las excluimos como parte de la selección de datos, porque usarlas implicaría **fuga de información** (*data leakage*): el modelo recibiría como entrada la respuesta o una versión de ella.

| Variable | Fuente | Motivo |
|---|---|---|
| `LogMass`, `log_SFR_Ha`, `log_SFR_ssp`, `log_age_mean_LW`, `log_ZH_mean_LW`, `vel_sigma_Re` | Pipe3D | Salen del mismo ajuste espectral que las etiquetas |
| `LogMassNSA`, `LogMassTaylor`, `nsa_sersic_mass` | NSA y Taylor | Son otras estimaciones de la masa. Se usan **solo para validar** nuestras mediciones |
| `error_log_mstar`, `error_log_sfr`, `error_d4000` | Pipe3D | Describen la etiqueta, no la galaxia |
| `QCFLAG` | Pipe3D | Es constante (0) en la muestra |
| `mangaid`, RA, Dec | Catálogo | Son identificadores. Con las coordenadas en bruto un modelo podría memorizar zonas del cielo; la información útil de la posición (telescopio y extinción) la construimos aparte |

In [ ]:
cat = pd.read_csv(CATALOGO, dtype={"mangaid": str})
print("Catálogo completo:", len(cat), "galaxias")
cat = cat[cat.QCFLAG_original == 0]
datos = cat.rename(columns={"RA_Pipe3D": "ra", "DEC_Pipe3D": "dec", "split_proyecto": "particion", "redshift": "z"})[
    ["mangaid", "ra", "dec", "z", "particion", *OBJETIVOS]].reset_index(drop=True)

# Catálogo auxiliar del repositorio: -999 marca valores ausentes.
aux = Table.read(REPO_DIR / "Datos" / "inferencia.fits").to_pandas()
aux["mangaid"] = aux.MANGAID.map(lambda x: x.decode() if isinstance(x, bytes) else x).str.strip()
aux = aux[["mangaid", "T_11", "PETRO_TH90", "C", "A", "LogMassNSA", "nsa_sersic_ba"]].replace(-999.0, np.nan)
aux = aux.rename(columns={"C": "C_catalogo", "A": "A_catalogo"})
datos = datos.merge(aux, on="mangaid", how="left", validate="one_to_one")

print("Con QCFLAG = 0:", len(datos), "·", datos.particion.value_counts().to_dict())
print("Sin redshift:", datos.z.isna().sum(), "· sin T_11:", datos.T_11.isna().sum())

### 4.1 Auditoría de las entradas

Antes de medir nada revisamos que cada imagen de la caché sea utilizable. Las comprobaciones vienen del cuaderno de Entrenamiento (mejora M04):

| Comprobación | Por qué |
|---|---|
| Todos los píxeles son números finitos | Un `NaN` o un infinito se propaga a todas las mediciones y a la pérdida de la red |
| Ninguna banda está completamente en cero | Una banda en cero significa que no hay datos en esa banda. El color de la galaxia saldría falso. En el experimento anterior encontramos tres casos: `1-290015`, `1-390056` y `1-626493` |
| No hay imágenes idénticas en particiones distintas | Si la misma imagen estuviera en entrenamiento y en validación, el error de validación saldría optimista (fuga de información). Comparamos una huella (SHA-1) de cada recorte |

Excluimos las galaxias que fallan las dos primeras comprobaciones, y de cada par duplicado entre particiones conservamos solo la copia de entrenamiento. La exclusión no se basa en las etiquetas ni en los errores del modelo.

En el mismo recorrido por la caché calculamos la grumosidad de cada galaxia (sección 6.7), para no leer dos veces los 5 GB de imágenes. Con muchos procesadores tarda unos 5 minutos; con 2, hasta 25. Si Colab se desconecta, continúa desde el último bloque guardado.

In [ ]:
t0 = time.time()
EQUIPO = IE.recorrer_cache_equipo(CACHE, RESULTADOS / "mediciones_equipo", guard,
                                  ids_validos=set(datos.mangaid), workers=os.cpu_count())
print(f"Recorrido de la caché: {len(EQUIPO)} galaxias en {time.time() - t0:.0f} s")

aud = datos[["mangaid", "particion"]].merge(EQUIPO, on="mangaid", how="left")
con_imagen = aud.huella.notna()
no_finita = con_imagen & ~aud.finito.fillna(True).astype(bool)
banda_cero = con_imagen & aud.banda_en_cero.fillna(False).astype(bool)
# Duplicados exactos: misma huella en más de una partición. Se conserva la copia de entrenamiento.
rep = aud[con_imagen].groupby("huella").particion.agg(lambda p: set(p))
cruzadas = rep[rep.map(len) > 1].index
dup_fuera = aud.huella.isin(cruzadas) & (aud.particion != "Train")
display(pd.Series({"galaxias con QCFLAG = 0": len(aud),
                   "sin imagen en la caché": int((~con_imagen).sum()),
                   "con píxeles no finitos": int(no_finita.sum()),
                   "con una banda completa en cero": int(banda_cero.sum()),
                   "imágenes idénticas en particiones distintas": len(cruzadas)}, name="galaxias").to_frame())
EXCLUIDAS_AUDITORIA = aud.loc[no_finita | banda_cero | dup_fuera, ["mangaid", "particion"]]
if len(EXCLUIDAS_AUDITORIA):
    display(EXCLUIDAS_AUDITORIA)
datos = datos[~datos.mangaid.isin(EXCLUIDAS_AUDITORIA.mangaid)].reset_index(drop=True)
print("Galaxias que siguen:", len(datos), "·", datos.particion.value_counts().to_dict())

**Cómo leer el resultado.** Esperábamos encontrar las tres galaxias con una banda en cero que había registrado el experimento anterior, y ningún duplicado exacto, porque cada galaxia de MaNGA tiene su propio recorte. La tabla dice cuántas encontramos realmente. Estas comprobaciones no detectan solapamientos parciales entre galaxias cercanas en el cielo; esa limitación también la señalaba el cuaderno de Entrenamiento.

## 5. Preparación de la imagen para la red

La red no recibe el FITS tal como se descarga. Antes de entrar a la ResNet50 (He et al. 2016), cada imagen pasa por una serie de pasos que son, en sí mismos, ingeniería de características a nivel de píxel. Los definimos al reproducir Katachi; en esta sección los documentamos con las mismas categorías de la rúbrica y los medimos sobre la caché real.

| Paso | Qué se hace | Técnica | Por qué |
|---|---|---|---|
| Recorte | Cuadro central de 300 × 300 px (78.6″) del FITS de 800 px, a la escala nativa de 0.262″/px y sin remuestrear | Reducción de dimensionalidad | Conserva la galaxia y reduce la entrada de 1.9 M a 270 mil valores. No se remuestrea para no interpolar píxeles |
| Estiramiento | asinh(flujo / (β · σ_b)), con β = 3 y σ_b el ruido del cielo en cada banda | Transformación no lineal | El flujo tiene un rango enorme entre el núcleo y el halo. El asinh se comporta como lineal cerca del ruido y como logaritmo en la luz brillante (Lupton, Gunn & Szalay 1999), así que comprime la cola sin perder la luz débil |
| Escala por banda | División entre s_b, el percentil 99.9 de los píxeles de entrenamiento ya estirados | Escalamiento (tipo min–max con un máximo robusto) | Deja las tres bandas en un rango comparable, con el cielo en 0 y la galaxia hasta ~1. El percentil 99.9 evita que unas pocas estrellas brillantes fijen la escala |
| Orden de bandas | De g, r, z (orden del FITS) a z, r, g | Codificación de canales | La primera capa de la ResNet50 viene entrenada con ImageNet (Deng et al. 2009), donde el primer canal es el rojo. Así la banda más roja entra por ese canal, como en Katachi |
| Planos de la cadena | La red de SFR recibe un cuarto canal constante con la masa estimada, y la de D4000 un quinto con la SFR estimada | Construcción de características | Es la idea central de Katachi: cada propiedad se estima usando las anteriores |
| Aumentación | Reflejos y rotaciones aleatorias, solo al entrenar | Generación de ejemplos | Una galaxia girada sigue siendo la misma; la red aprende a no depender de la orientación |

Katachi original partía de imágenes PNG de SDSS, ya convertidas a color de 0 a 255, y solo las dividía entre 255. Con FITS en nanomaggies esa conversión no existe, así que la reemplazamos por los dos primeros pasos.

**Qué agrega el equipo en esta sección.** El cuaderno de Preprocesamiento exploró, con una galaxia de prueba (NGC 383), cómo estimar el fondo del cielo, cómo combinar las bandas y qué estiramiento conserva mejor la luz débil. El de Entrenamiento propuso otras representaciones de las bandas para la red y estandarizar las etiquetas. Aquí llevamos esas ideas a nuestros datos y las medimos en muchas galaxias:

| Subsección | Pregunta | Origen |
|---|---|---|
| 5.1 | ¿Qué constantes usa hoy la red? | Reproducción de Katachi |
| 5.2 | ¿Hace falta un mapa de fondo del cielo, o basta un valor constante? | Preprocesamiento |
| 5.3 | ¿Qué estiramiento elegimos, y para qué uso? | Preprocesamiento + prueba nueva |
| 5.4 | ¿Cómo quedan los píxeles con cada representación? | Reproducción de Katachi + Entrenamiento |
| 5.5 | ¿Qué escala tienen los planos, las etiquetas y el redshift? | Reproducción de Katachi + Entrenamiento |
| 5.6 | ¿Qué decisiones quedan para la fase de modelado? | Entrenamiento |

En esta sección no entrenamos ninguna red. Cargamos una muestra de galaxias de entrenamiento de la caché y medimos cómo cambia la distribución de los píxeles en cada paso, usando el mismo código que usa la red (`src/katachi_fits/transforms.py`).

### 5.1 Constantes de la transformación

σ_b y s_b se calcularon una sola vez con galaxias de entrenamiento, al construir la caché, y se usan igual para todas las galaxias. La siguiente celda las lee del archivo que guardó esa corrida. Si no lo encuentra, usa los valores que quedaron registrados en ella.

In [ ]:
import torch
from src.katachi_fits.transforms import Normalizer, preprocess

DIR_NORM = BASE / KC.RESULTS_DIRNAME / "normalizacion"
f_norm = sorted(p for p in DIR_NORM.glob("normalizador_recorte300_beta3*.json") if "_lim" not in p.name) if DIR_NORM.exists() else []
if f_norm:
    NORM = Normalizer.from_json(f_norm[0].read_text()); fuente_norm = f_norm[0].name
else:   # valores registrados en la corrida de la red
    NORM = Normalizer(np.array([0.00209, 0.00396, 0.01075]), np.array([5.709, 6.027, 5.506]), 3.0)
    fuente_norm = "valores registrados en la corrida de la red"
display(pd.DataFrame({"σ_cielo [nmgy/px]": NORM.sigma, "s_b": NORM.scale}, index=list(FE.BANDAS)).round(5))
print("β =", NORM.beta, "· fuente:", fuente_norm)

# Muestra de galaxias de entrenamiento leída directamente de la caché
ids_tr = set(datos.loc[datos.particion == "Train", "mangaid"])
N_PIX_GAL = int(env("AV2_N_PIX_GAL", "200"))
X_PIX, ids_pix = [], []
for f in sorted(CACHE.glob("chunk_*.npz")):
    with np.load(f) as z:
        sel = [i for i, m_ in enumerate(z["mangaid"].astype(str)) if m_ in ids_tr]
        X_PIX.append(z["x"][sel[:N_PIX_GAL - len(ids_pix)]].astype(np.float32))
        ids_pix += list(z["mangaid"].astype(str)[sel[:N_PIX_GAL - len(ids_pix)]])
    if len(ids_pix) >= N_PIX_GAL:
        break
X_PIX = np.concatenate(X_PIX)
print("Galaxias de entrenamiento en la muestra:", len(X_PIX))

### 5.2 Fondo del cielo y fuentes vecinas

**Qué es el fondo del cielo.** Aunque DESI ya resta el cielo al combinar las exposiciones, siempre queda un nivel residual: luz de la atmósfera, de la Luna o del polvo del Sistema Solar. Si no se resta, se suma a la galaxia, sobre todo en sus partes débiles. Hay dos formas de estimarlo:

- **un valor constante por banda**, medido en el anillo exterior del campo. Es lo que hace la red (sección 5.1) y lo que hacemos al medir las características (sección 6);
- **un mapa** que cambia de una zona a otra de la imagen, con `Background2D` de photutils (Bradley et al.), que calcula la mediana con recorte sigma en cajas y luego interpola (Bertin & Arnouts 1996). Es la propuesta del cuaderno de Preprocesamiento.

**Qué medimos.** Tomamos `N_DEMO_FITS` cubos completos de 800 × 800 px de galaxias de entrenamiento con tamaños distintos y comparamos el flujo de la galaxia en la banda r, dentro de 2 R90, con tres estimaciones del fondo:

1. **constante del anillo**: la referencia;
2. **mapa con la caja original del Preprocesamiento**: caja de 1/5 del eje mayor de la galaxia, sin enmascarar;
3. **mapa adaptado**: caja de 4 R90 (entre 32 y 200 px) y la galaxia y las estrellas enmascaradas antes de estimar el fondo.

También construimos la máscara gaussiana del Preprocesamiento sobre la suma de las bandas (galaxia central y demás fuentes) y la comparamos con la máscara que usa el resto del cuaderno: la del equipo, en la banda r (`src/data/mask_eval.py`). Por último, ajustamos su borde con un contorno activo (Caselles et al. 1997; Márquez-Neila et al. 2014).

In [ ]:
from src.katachi_fits.cache import read_full
from src.caracteristicas_desi.fotometria import cielo, segmentar, mapa_radial

# FITS de 800 px en Drive. Si no están disponibles, se usan los recortes de 300 px de la caché.
DIR_FITS = DESCARGA / "fits"
ARCHIVOS = {}
if DIR_FITS.exists():
    for nombre in os.listdir(DIR_FITS):
        if nombre.endswith((".fits", ".fits.gz")):
            ARCHIVOS.setdefault(nombre.split("_")[0], DIR_FITS / nombre)
cand = datos[datos.mangaid.isin(ARCHIVOS) & datos.PETRO_TH90.notna() & (datos.particion == "Train")]
if len(cand) < N_DEMO_FITS:
    cand = datos[datos.mangaid.isin(ARCHIVOS) & datos.PETRO_TH90.notna() & (datos.particion != "Test")]
cand = cand.sort_values("PETRO_TH90")
DEMO = cand.iloc[np.linspace(0, len(cand) - 1, min(N_DEMO_FITS, len(cand))).astype(int)] if len(cand) else cand
if len(DEMO):
    cubos_demo = [read_full(ARCHIVOS[m]) for m in DEMO.mangaid]
    print(f"Cubos completos de {cubos_demo[0].shape[-1]} px:", len(cubos_demo))
else:
    print("No se encontraron los FITS: usamos recortes de 300 px de la caché.")
    p90 = datos.set_index("mangaid").PETRO_TH90
    DEMO = pd.DataFrame({"mangaid": ids_pix[:N_DEMO_FITS], "PETRO_TH90": [p90.get(m, np.nan) for m in ids_pix[:N_DEMO_FITS]]})
    cubos_demo = list(X_PIX[:N_DEMO_FITS])

def flujo_apertura(img, radio_px):
    return float(img[mapa_radial(img.shape[0]) < radio_px].sum())

filas, figs = [], []
for (_, g), cubo in zip(DEMO.iterrows(), cubos_demo):
    try:
        n = cubo.shape[-1]
        r90_px = g.PETRO_TH90 / FE.PIXSCALE if np.isfinite(g.PETRO_TH90) else 20.0
        radio = min(2 * r90_px, (n - 1) / 2 - 1)
        r = cubo[1]
        # 1. Constante del anillo exterior (como la red): r > 300 px en 800 px; esquinas en 300 px.
        med_r, sig_r = cielo(r, r_min_px=KC.SKY_RING_PX if n >= 800 else FE.CIELO_R_MIN_PX)
        f_ref = flujo_apertura(r - med_r, radio)
        # Máscara de fuentes para el mapa adaptado: galaxia y estrellas, detectadas con la constante anterior.
        gal0, otras0 = IE.mascara_gaussiana(r - med_r, sig_r)
        fuentes = ndimage.binary_dilation(gal0 | otras0, iterations=5)
        # 2. Mapa con la caja original (eje mayor / 5) y sin máscara.
        caja_orig = int(max(8, round(2 * r90_px / 5)))
        b_orig = IE.fondo_2d(r, caja_orig)
        # 3. Mapa adaptado.
        caja_ad = IE.caja_fondo(g.PETRO_TH90, n)
        b_ad = IE.fondo_2d(r, caja_ad, mascara=fuentes)
        f_orig = flujo_apertura(r - b_orig.background, radio)
        f_ad = flujo_apertura(r - b_ad.background, radio)
        # Máscaras sobre la parte central (400 px como máximo, para que sea rápido).
        c0 = max(0, n // 2 - 200); cen = slice(c0, n - c0)
        med_sig = [cielo(cubo[b], r_min_px=KC.SKY_RING_PX if n >= 800 else FE.CIELO_R_MIN_PX) for b in range(3)]
        sub = (cubo - np.array([m for m, _ in med_sig], np.float32)[:, None, None])[:, cen, cen]
        suma = sub.mean(0)
        _, _, sig_suma = sigma_clipped_stats(suma, sigma=3, maxiters=5)
        gal_g, otras_g = IE.mascara_gaussiana(suma, sig_suma)
        gal_eq, _, _ = segmentar(sub[1])
        vis = IE.normalizar(suma, "log", IE.PercentileInterval(98))
        gal_ca = IE.contorno_activo(vis, gal_g)
        iou = (gal_g & gal_eq).sum() / max((gal_g | gal_eq).sum(), 1)
        filas.append({"mangaid": g.mangaid, "PETRO_TH90 [″]": g.PETRO_TH90, "σ cielo r (anillo)": sig_r,
                      "rms del mapa r (mediana)": float(np.median(b_ad.background_rms)),
                      "caja original [px]": caja_orig, "caja adaptada [px]": caja_ad,
                      "Δ flujo, caja original [%]": 100 * (f_orig - f_ref) / f_ref,
                      "Δ flujo, mapa adaptado [%]": 100 * (f_ad - f_ref) / f_ref,
                      "IoU máscara gaussiana vs. del equipo": iou,
                      "área gaussiana / del equipo": gal_g.sum() / max(gal_eq.sum(), 1),
                      "área contorno activo / gaussiana": gal_ca.sum() / max(gal_g.sum(), 1),
                      "fuentes vecinas": int(ndimage.label(otras_g)[1])})
        figs.append((g.mangaid, sub, b_ad.background[cen, cen] - med_r, vis, gal_g, otras_g, gal_eq, gal_ca))
    except Exception as e:      # una galaxia problemática no detiene la sección
        print(f"{g.mangaid}: no se pudo procesar ({type(e).__name__}: {str(e)[:80]})")
FONDO = pd.DataFrame(filas).set_index("mangaid")
display(FONDO.round(3))
print(f"Cambio mediano del flujo en r: caja original {FONDO['Δ flujo, caja original [%]'].median():+.1f} % · "
      f"mapa adaptado {FONDO['Δ flujo, mapa adaptado [%]'].median():+.1f} % (frente a la constante del anillo)")
print(f"Diferencia típica entre el mapa y la constante: {np.median(np.abs(figs[0][2])):.1e} nmgy/px "
      f"(σ del cielo en r: {FONDO['σ cielo r (anillo)'].iloc[0]:.1e})")
print(f"IoU mediano entre la máscara gaussiana y la del equipo: {FONDO['IoU máscara gaussiana vs. del equipo'].median():.2f} · "
      f"área gaussiana / del equipo: {FONDO['área gaussiana / del equipo'].median():.2f}")

In [ ]:
assert figs, "Ninguna galaxia de la sección 5.2 se pudo procesar"
fig, ax = plt.subplots(min(3, len(figs)), 4, figsize=(14, 3.5 * min(3, len(figs))), squeeze=False)
for k, (m, sub, mapa, vis, gal_g, otras_g, gal_eq, gal_ca) in enumerate(figs[:3]):
    ax[k, 0].imshow(make_lupton_rgb(sub[2], sub[1], sub[0], Q=8, stretch=0.05), origin="lower")
    ax[k, 0].set_title(f"{m} · z, r, g", fontsize=8)
    im = ax[k, 1].imshow(mapa, origin="lower", cmap="coolwarm", vmin=-np.abs(mapa).max(), vmax=np.abs(mapa).max())
    ax[k, 1].set_title("mapa de fondo − constante (r) [nmgy]", fontsize=8); plt.colorbar(im, ax=ax[k, 1], shrink=0.8)
    ax[k, 2].imshow(vis, origin="lower", cmap="gray")
    ax[k, 2].imshow(np.ma.masked_where(~otras_g, otras_g), origin="lower", cmap="autumn", alpha=0.6)
    ax[k, 2].contour(gal_g, [0.5], colors="cyan", linewidths=0.8)
    ax[k, 2].contour(gal_eq, [0.5], colors="white", linewidths=0.8, linestyles="--")
    ax[k, 2].set_title("gaussiana (cian), del equipo (blanco), otras fuentes (rojo)", fontsize=8)
    ax[k, 3].imshow(vis, origin="lower", cmap="gray")
    ax[k, 3].contour(gal_ca, [0.5], colors="yellow", linewidths=0.8)
    ax[k, 3].set_title("contorno activo", fontsize=8)
    for a in ax[k]:
        a.set_xticks([]); a.set_yticks([])
plt.suptitle("Fondo del cielo y máscaras (adaptado del cuaderno de Preprocesamiento)", y=1.0)
plt.tight_layout(); plt.show()

**Por qué lo hacemos.** El fondo y las máscaras se aplican antes que todo lo demás, así que un error aquí pasa a todas las características y a la imagen que recibe la red. Antes de cambiar la forma de restar el cielo queríamos saber si el cambio importa.

**Cómo leer el resultado.** Una caja más pequeña que la galaxia y sin máscara confunde el halo de la galaxia con cielo y lo resta: el flujo en la columna "caja original" debería salir menor que con la constante. Enmascarar la galaxia y usar una caja más grande corrige eso. En nuestras pruebas locales con seis cubos, la caja original quitó alrededor del 40 % del flujo de la galaxia, y el mapa adaptado cambió el flujo en un 1 % o menos, con diferencias frente a la constante de una pequeña fracción de σ del cielo. Si la tabla de arriba muestra lo mismo, una constante por banda es suficiente para nuestros recortes de 78.6″: es la opción más simple y la que siguen usando la red y la sección 6. El mapa de fondo queda como opción para campos más grandes o con gradientes de luz, por ejemplo cerca de una estrella brillante.

Las dos máscaras se parecen en la forma, pero no en el tamaño: la del equipo, con umbral de 2σ en la banda r, llega más lejos en el halo que la gaussiana; por eso el IoU es moderado. Como la sección 6 necesita toda la luz de la galaxia, conservamos la máscara del equipo, que en la evaluación de máscaras recuperaba el 99.9 % del flujo. De la gaussiana nos quedamos con la idea de marcar todas las fuentes del campo, que es lo que hace el relleno de vecinos de la sección 6. El contorno activo ajusta el borde a la luz visible. Si cambia poco el área (última columna cerca de 1), no hace falta agregarlo al flujo, y así lo dejamos.

### 5.3 Colapso de bandas y estiramiento: qué normalización usar y para qué

Una imagen astronómica tiene un rango enorme: el núcleo puede ser miles de veces más brillante que el halo. Para verla, o para que una red la use, hay que **estirarla**, es decir, aplicarle una función que comprima los valores altos. El cuaderno de Preprocesamiento comparó, con una galaxia, 27 combinaciones:

- **3 formas de juntar las bandas:** promedio con el cielo restado, promedio crudo (como control) y promedio pesado por el inverso de la varianza (la banda con menos ruido pesa más);
- **3 estiramientos:** lineal, logarítmico (a = 100) y asinh (que pasa de lineal a logarítmico a 3σ del cielo);
- **3 intervalos de corte**, calculados en cada imagen: mínimo-máximo, percentil 98 y percentil asimétrico 1–98.

Y las calificó con tres métricas de imagen:

- la **entropía de Shannon** (Shannon 1948), en bits: cuántos tonos distintos se aprovechan;
- la **saturación**: qué porcentaje de la galaxia queda en el valor máximo;
- el **ancho de la CDF**: cuántos niveles de gris hay entre los percentiles 15 y 85.

**Lo que agregamos.** Las tres métricas miden qué tan bien se *ve* la galaxia. Pero el cuaderno de Entrenamiento advierte, con razón, que no conviene elegir la entrada de la red por su aspecto: un intervalo calculado en cada imagen lleva **todas** las galaxias al rango [0, 1], y con eso borra la diferencia de brillo entre una galaxia y otra. Para segmentar no importa, pero para estimar la masa sí, porque una galaxia más masiva es más brillante. Para medirlo agregamos una cuarta métrica, el **orden de brillo**: la correlación de Spearman (ρ), entre galaxias, del brillo central normalizado con log M\*. Como referencias incluimos el flujo crudo y la transformación global de la red, que es la misma para todas las galaxias.

Repetimos la comparación en las galaxias de entrenamiento de la muestra de la sección 5.1, en lugar de en una sola.

In [ ]:
t0 = time.time()
REJ = IE.rejilla_normalizaciones(X_PIX, ids_pix, (NORM.sigma, NORM.scale, NORM.beta))
REJ = REJ.merge(datos[["mangaid", "log_mstar"]], on="mangaid")
filas = []
for clave, g in REJ.groupby(["colapso", "estiramiento", "intervalo"], sort=False):
    filas.append({"colapso": clave[0], "estiramiento": clave[1], "intervalo": clave[2],
                  "entropía [bits]": g["entropía [bits]"].mean(), "saturación [%]": g["saturación [%]"].mean(),
                  "ancho CDF": g["ancho CDF"].mean(),
                  # Si el brillo central es igual en todas las galaxias (todas saturadas), no ordena nada: ρ = 0.
                  "ρ(brillo central, log M*)": (stats.spearmanr(g["brillo central"], g.log_mstar)[0]
                                                if g["brillo central"].nunique() > 1 else 0.0),
                  "galaxias con el centro saturado [%]": 100 * (g["brillo central"] >= 0.999).mean()})
TAB_NORM = pd.DataFrame(filas).set_index(["colapso", "estiramiento", "intervalo"])
print(f"{REJ.mangaid.nunique()} galaxias · {len(TAB_NORM) - 2} combinaciones · {time.time() - t0:.0f} s")

# Criterio del Preprocesamiento: mayor ancho de la CDF, luego mayor entropía, luego menor saturación.
grid = TAB_NORM.drop(index="referencia", level="colapso")
orden = grid.sort_values(["ancho CDF", "entropía [bits]", "saturación [%]"], ascending=[False, False, True])
print("Las cinco mejores con las métricas de imagen del Preprocesamiento:")
display(orden.head(5).round(3))
print("Referencias:")
display(TAB_NORM.xs("referencia", level="colapso").round(3))
ganadora = orden.index[0]
rho = TAB_NORM["ρ(brillo central, log M*)"]
print(f"Ganadora por métricas de imagen: {' · '.join(ganadora)} → ρ con log M* = {rho[ganadora]:.2f}")
print(f"Transformación global de la red: ρ = {rho[('referencia', 'global de la red (Katachi)', 'el mismo para todas')]:.2f} · "
      f"flujo crudo: ρ = {rho[('referencia', 'flujo crudo', 'ninguno')]:.2f}")
print(f"Mediana de ρ en las 27 combinaciones con intervalo por imagen: {grid['ρ(brillo central, log M*)'].median():.2f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for est, mk in (("lineal", "s"), ("log", "o"), ("asinh", "^")):
    s = grid.xs(est, level="estiramiento")
    ax[0].scatter(s["entropía [bits]"], s["ρ(brillo central, log M*)"], marker=mk, label=est)
ax[0].axhline(rho[("referencia", "global de la red (Katachi)", "el mismo para todas")], color="C3", ls="--", label="global de la red")
ax[0].set(xlabel="entropía [bits] (calidad visual)", ylabel="ρ(brillo central, log M*)",
          title="Verse bien no es lo mismo que conservar el brillo"); ax[0].legend(fontsize=8)
m = ids_pix[int(np.argmax(X_PIX[:, 1].sum(axis=(1, 2))))]
c = IE.colapsos(X_PIX[ids_pix.index(m)])
ax[1].imshow(IE.normalizar(c[ganadora[0]], ganadora[1], IE._intervalos()[ganadora[2]]), origin="lower", cmap="gray")
ax[1].set_title(f"{m} con la ganadora: {' · '.join(ganadora)}", fontsize=8); ax[1].set_xticks([]); ax[1].set_yticks([])
plt.tight_layout(); plt.show()

**Por qué lo hacemos.** El cuaderno de Preprocesamiento eligió su normalización por la calidad de la imagen, pensando en la red de segmentación (Attention UNet). El de Entrenamiento usa una transformación fija para la red de regresión. Las dos posturas parecían contradecirse; con esta prueba vemos que responden a usos distintos.

**Cómo leer el resultado.** La entropía y el ancho de la CDF premian los intervalos calculados por imagen con estiramiento log o asinh: así se ve la estructura débil sin saturar el núcleo. Pero esas mismas combinaciones dejan el brillo central de casi todas las galaxias cerca del máximo, y su ρ con la masa cae muy por debajo del de la transformación global, que conserva el orden porque es la misma función para todas las galaxias. Algunas combinaciones de contraste bajo incluso invierten el orden (ρ negativo). Restar el cielo antes de promediar tampoco cambia las métricas, porque el intervalo calculado en cada imagen ya elimina cualquier constante. **Decisión:**

- para **máscaras y visualización** usamos un intervalo por imagen con estiramiento log o asinh, como los mejor calificados de la tabla (en la sección 5.2 usamos log con percentil 98, el ganador del cuaderno original);
- para la **red de regresión** conservamos una transformación global por banda (asinh con σ del cielo, sección 5.1, o la variante de Entrenamiento, sección 5.4), porque no borra la información de brillo que se relaciona con la masa.

El cálculo se hizo con la entropía en bits y la saturación como porcentaje de la galaxia. Al adaptar el código corregimos estos dos cálculos: el original usaba base 10 y contaba casi todos los píxeles distintos de cero como saturados.

### 5.4 Distribución de los píxeles con cada representación

Comparamos seis versiones de los mismos píxeles:

1. el flujo crudo, en nanomaggies;
2. después del estiramiento asinh;
3. después de dividir entre s_b, que es lo que recibe la red;
4. estandarizado a media 0 y desviación 1 por banda, que es la escala con que se usan normalmente los pesos de ImageNet. Esta versión no se aplicó en la red; la incluimos como referencia;
5. **la representación de bandas del cuaderno de Entrenamiento (M05):** asinh(f / s_b) con s_b el percentil 75 de |f| en entrenamiento, y después media 0 y desviación 1 con los estadísticos de entrenamiento;
6. **la representación RGB del cuaderno de Entrenamiento:** imagen de color de Lupton (R = z, G = r, B = g; Q = 8, stretch = 0.5) con la normalización de ImageNet. Se muestra en el orden g, r, z para compararla con las demás.

Para cada una medimos la asimetría, la curtosis, la media, la desviación y algunos percentiles.

In [ ]:
sig = NORM.sigma[None, :, None, None].astype(np.float32)
esc = NORM.scale[None, :, None, None].astype(np.float32)
etapas = {"1 · flujo crudo [nmgy]": X_PIX,
          "2 · asinh(x / βσ)": np.arcsinh(X_PIX / (NORM.beta * sig))}
etapas["3 · ÷ s_b (entrada de la red)"] = etapas["2 · asinh(x / βσ)"] / esc
# Comprobación: es exactamente la transformación que usa la red
red = preprocess(torch.from_numpy(X_PIX[:4]), NORM, X_PIX.shape[-1]).numpy()
assert np.allclose(red, etapas["3 · ÷ s_b (entrada de la red)"][:4], atol=1e-5)
m3 = etapas["3 · ÷ s_b (entrada de la red)"].mean(axis=(0, 2, 3), keepdims=True)
d3 = etapas["3 · ÷ s_b (entrada de la red)"].std(axis=(0, 2, 3), keepdims=True)
etapas["4 · estandarizado (referencia)"] = (etapas["3 · ÷ s_b (entrada de la red)"] - m3) / d3
# Representaciones del cuaderno de Entrenamiento (M05); la muestra es de entrenamiento.
etapas["5 · asinh(x / p75) + estandarización (Entrenamiento)"], ST_P75 = IE.asinh_p75(X_PIX, X_PIX)
etapas["6 · RGB de Lupton + ImageNet (Entrenamiento)"] = IE.rgb_lupton_imagenet(X_PIX)
display(pd.DataFrame(ST_P75, index=list(FE.BANDAS)).round(5).rename_axis("estadísticos de la etapa 5"))

filas = []
for etapa, A in etapas.items():
    for b, nb in enumerate(FE.BANDAS):
        v = rng.choice(A[:, b].ravel(), 1_000_000, replace=False)
        filas.append({"etapa": etapa, "banda": nb, "asimetría": stats.skew(v), "curtosis": stats.kurtosis(v),
                      "media": v.mean(), "desviación": v.std(), "p1": np.percentile(v, 1),
                      "mediana": np.median(v), "p99.9": np.percentile(v, 99.9), "máximo": v.max()})
TABLA_PIX = pd.DataFrame(filas).set_index(["etapa", "banda"])
display(TABLA_PIX.round(4))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
for a, etapa in zip(ax, list(etapas)[:3]):
    for b, nb in enumerate(FE.BANDAS):
        v = rng.choice(etapas[etapa][:, b].ravel(), 1_000_000, replace=False)
        lo, hi = np.percentile(v, [0.05, 99.99])
        a.hist(v, bins=200, range=(lo, hi), histtype="step", label=nb, log=True)
    a.set(title=etapa, xlabel="valor del píxel"); a.legend()
ax[0].set_ylabel("número de píxeles")
plt.suptitle("Distribución de los píxeles en cada paso (muestra de entrenamiento)", y=1.03)
plt.tight_layout(); plt.show()

k = int(np.argmax(X_PIX[:, 1].sum(axis=(1, 2))))          # la galaxia más brillante de la muestra
fig, ax = plt.subplots(1, 3, figsize=(12, 3.8))
for a, etapa in zip(ax, list(etapas)[:3]):
    im = a.imshow(etapas[etapa][k, 1], origin="lower", cmap="magma",
                  vmin=np.percentile(etapas[etapa][k, 1], 1), vmax=np.percentile(etapas[etapa][k, 1], 99.9))
    a.set_title(etapa, fontsize=9); a.set_xticks([]); a.set_yticks([]); plt.colorbar(im, ax=a, shrink=0.8)
plt.suptitle(f"{ids_pix[k]} · banda r: el estiramiento hace visible el halo sin saturar el núcleo", y=1.02)
plt.tight_layout(); plt.show()

**Por qué lo hacemos.** Una red neuronal aprende mal cuando sus entradas tienen valores muy dispares: si unos píxeles valen miles de veces más que otros, esos pocos dominan el aprendizaje y la luz débil de la galaxia casi no cuenta. Medir la distribución en cada paso nos permite comprobar que la preparación de la imagen corrige ese problema.

**Qué observamos.** El flujo crudo tiene una asimetría enorme (149 en g, 67 en r y 36 en z): casi todos los píxeles son cielo cercano a cero y unos pocos son muchísimo más brillantes. El estiramiento asinh la reduce a cerca de 3 en las tres bandas, y la división entre s_b deja los valores entre 0 y aproximadamente 1. Estandarizar (paso 4) solo cambia la escala, no la forma, así que el paso que realmente corrige la asimetría es el asinh. En la imagen de ejemplo se ve el efecto: con el flujo crudo solo se distinguen los puntos más brillantes, y después del asinh aparecen el halo de la galaxia y las fuentes débiles sin que el centro se sature. En esa galaxia también aparece una franja brillante en el borde superior, que es un defecto de la imagen; los pequeños picos en el extremo derecho de los histogramas vienen de píxeles saturados como esos.

**Las representaciones del cuaderno de Entrenamiento.** En la etapa 5, s_b (el percentil 75 de |f|) sale menor que 3σ del cielo (compárese con la tabla de la sección 5.1). Por eso la compresión empieza a un brillo más bajo y la asimetría baja más que con el asinh de la red, de cerca de 3 a cerca de 1.5 en nuestra prueba local. Después de estandarizar, las tres bandas quedan con media 0 y desviación 1, la escala que esperan los pesos de ImageNet.

La etapa 6 sigue el camino de Katachi con SDSS: imágenes de color de 8 bits. Su ventaja es que se parece a lo que vio la red preentrenada. Pero con Q = 8 y stretch = 0.5 aplicados a flujos en nanomaggies, casi todos los píxeles quedan en negro (la mediana es el valor del cero) y la asimetría sigue alta. Antes de usarla habría que ajustar el stretch a nuestras unidades; lo anotamos para la fase de modelado. Qué representación funciona mejor para estimar las propiedades se medirá con la variante D frente a C del plan de modelado (sección 15).

### 5.5 Escala de los planos, las etiquetas y el redshift

Las redes de SFR y D4000 reciben, además de la imagen, un plano constante con la masa y otro con la SFR estimadas, sin normalizar. Comparamos su escala con la de los canales de imagen. Si un canal vale cerca de 10 mientras los demás valen cerca de 0.1, al inicio del entrenamiento sus pesos reciben gradientes mucho mayores, lo que puede explicar los picos de la pérdida de la SFR que observamos.

Después aplicamos la estandarización que propone el cuaderno de Entrenamiento (M09): a cada etiqueta, y al redshift, se le resta su media de entrenamiento y se divide entre su desviación de entrenamiento. Al predecir se invierte la operación, así que los errores se siguen reportando en dex.

In [ ]:
canal = TABLA_PIX.xs("3 · ÷ s_b (entrada de la red)", level="etapa")
trn = datos[datos.particion == "Train"]
planos = pd.DataFrame({
    "media": [*canal["media"], trn.log_mstar.mean(), trn.log_sfr.mean()],
    "desviación": [*canal["desviación"], trn.log_mstar.std(), trn.log_sfr.std()],
}, index=[f"imagen {b}" for b in FE.BANDAS] + ["plano log M* (red de SFR y D4000)", "plano log SFR (red de D4000)"])
planos["|media| / desviación media de la imagen"] = planos["media"].abs() / canal["desviación"].mean()
display(planos.round(3))

# Estandarización con entrenamiento (Entrenamiento, M09)
ESCALA_ETIQUETAS = pd.DataFrame({c: {"media de entrenamiento": trn[c].mean(), "desviación de entrenamiento": trn[c].std()}
                                 for c in [*OBJETIVOS, "z"]}).T
display(ESCALA_ETIQUETAS.round(4))
canal5 = TABLA_PIX.xs("5 · asinh(x / p75) + estandarización (Entrenamiento)", level="etapa")
estd = lambda c: (trn[c] - ESCALA_ETIQUETAS.loc[c, "media de entrenamiento"]) / ESCALA_ETIQUETAS.loc[c, "desviación de entrenamiento"]
planos_est = pd.DataFrame({
    "media": [*canal5["media"], estd("log_mstar").mean(), estd("log_sfr").mean(), estd("z").mean()],
    "desviación": [*canal5["desviación"], estd("log_mstar").std(), estd("log_sfr").std(), estd("z").std()],
}, index=[f"imagen {b} (etapa 5)" for b in FE.BANDAS] + ["plano log M* estandarizado", "plano log SFR estandarizado", "redshift estandarizado"])
planos_est["|media| / desviación media de la imagen"] = planos_est["media"].abs() / canal5["desviación"].mean()
display(planos_est.round(3).rename_axis("con la estandarización del cuaderno de Entrenamiento"))

**Por qué lo hacemos.** Las redes de SFR y D4000 reciben, además de la imagen, la masa y la SFR estimadas. Si estos datos están en una escala muy distinta a la de la imagen, la red les da un peso desproporcionado al inicio del entrenamiento.

**Qué observamos.** Los canales de imagen valen en promedio 0.06, mientras que el plano de masa vale cerca de 10.6, unas 80 veces más, y el de SFR cerca de −0.8. Esa diferencia es consistente con la inestabilidad que tuvo la red al inicio de su entrenamiento, y justifica la mejora propuesta de estandarizar estos planos.

**Con la estandarización del cuaderno de Entrenamiento**, la segunda tabla muestra que los planos y el redshift quedan en la misma escala que la imagen (media 0, desviación 1), así que ninguna entrada domina el inicio del entrenamiento. La tabla de medias y desviaciones de entrenamiento es la que se guarda para invertir la transformación al predecir.

**Siguientes pasos para la imagen.** Además de esta estandarización, la propuesta de mejoras del equipo (`Documentación/Propuesta_mejoras_preprocesamiento_DESI.md`) plantea, como opciones que se activan una a una, agregar ruido y desenfoque de PSF (el desenfoque que produce la atmósfera) durante el entrenamiento, y normalizar por región norte/sur si la sección 11 confirma diferencias. Su efecto se medirá en la fase de modelado (sección 5.6).

### 5.6 Decisiones para la red que pasan a la fase de modelado

El cuaderno de Entrenamiento propone cambios a cómo se prepara la entrada de la red. Este avance no entrena redes, así que aquí dejamos registrado qué decidimos, con qué evidencia de este cuaderno y qué se va a comparar en la siguiente fase.

| Decisión | Katachi (lo que usa hoy la red) | Propuesta del equipo (Entrenamiento) | Evidencia en este cuaderno |
|---|---|---|---|
| Ajustar estadísticos | σ y s_b con entrenamiento | Todo con entrenamiento, también la escala de las etiquetas (M01) | El pipeline de la sección 10.3 ya se ajusta solo con entrenamiento |
| Representación de las bandas | asinh(f / 3σ) ÷ s_b | asinh(f / p75) y estandarización, o RGB de Lupton con ImageNet (M05) | Sección 5.4 |
| Etiquetas y planos de la cadena | Sin estandarizar | Etiquetas estandarizadas con entrenamiento; pérdida Huber como alternativa (M09) | Sección 5.5: corrige la diferencia de escala de 80 veces |
| Redshift | No se usa | Se agrega después del extractor visual (M08) | Sección 14: el redshift explica la mayor parte de la ventaja de las características en masa |
| Aumentos | Reflejos y rotación arbitraria con relleno 0 | Reflejos y giros de 90° (sin interpolar ni rellenar) | Celda siguiente |
| Arquitectura | Tres ResNet50 en cadena | Un extractor con tres salidas y ajuste fino gradual (M06, M07) | Se compara en la fase de modelado |

La celda siguiente mide cuántos píxeles rellena con ceros una rotación arbitraria del recorte de 300 px.

In [ ]:
ROT = IE.fraccion_relleno_rotacion(FE.RECORTE_PX, 5)
print(f"Rotación arbitraria: rellena en promedio el {100 * ROT['fracción rellenada'].mean():.1f} % de los píxeles "
      f"(hasta {100 * ROT['fracción rellenada'].max():.1f} % a 45°); giros de 90°: "
      f"{100 * ROT.loc[ROT['ángulo'] % 90 == 0, 'fracción rellenada'].max():.1f} %")
fig, ax = plt.subplots(figsize=(6, 2.6))
ax.plot(ROT["ángulo"], 100 * ROT["fracción rellenada"])
ax.set(xlabel="ángulo de rotación [°]", ylabel="píxeles rellenados [%]", title="Rotación del recorte de 300 px con relleno 0")
plt.tight_layout(); plt.show()

**Por qué lo hacemos.** Con una rotación arbitraria, las esquinas del recorte quedan en cero exacto, un valor que no aparece en el cielo real (el cielo tiene ruido). La red podría aprender a reconocer esos triángulos en lugar de la galaxia. Con giros de 90° no se pierde ni se inventa ningún píxel. A cambio, hay menos orientaciones distintas, y por eso la comparación se hará entrenando, en la siguiente fase.

## 6. Construcción de características

Sobre cada recorte (3 bandas, 300 × 300 píxeles, 0.262″/px, en nanomaggies) medimos las cantidades que se usan habitualmente en astronomía extragaláctica para describir galaxias. En lugar de proponer variables nuevas, partimos de definiciones publicadas, con interpretación física y comparables con otros catálogos. Todas se miden en el **mismo recorte que recibe la red**, para que la comparación de la sección 14 sea equivalente.

### 6.1 Cómo se mide cada galaxia

| Paso | Qué se hace | Por qué |
|---|---|---|
| Cielo residual | Mediana y desviación con recorte σ en las esquinas del recorte | DESI ya resta el cielo, pero queda un residuo que afecta las magnitudes de las galaxias débiles |
| Máscara de la galaxia | Umbral de 2σ sobre la banda r suavizada; componente que contiene el centro, sin huecos | Es la máscara que elegimos en la evaluación de máscaras del equipo (recupera 0.999 de la luz con 2 % de contaminación) |
| Vecinos | Fuentes detectadas a 3σ fuera de la galaxia; sus píxeles se reemplazan por el píxel simétrico respecto al centro | Una estrella o galaxia vecina aumentaría el flujo y alteraría el color. El relleno aprovecha que las galaxias son aproximadamente simétricas |
| Radio de Petrosian R_P | Radio donde el brillo del anillo [0.8R, 1.25R] es 0.2 veces el brillo medio interior | Es la definición de SDSS (Blanton et al. 2001). No depende de la profundidad de la imagen ni de la distancia, y la apertura de 2 R_P contiene casi todo el flujo |
| Flujo y magnitud | Flujo dentro de 2 R_P **medido en r** y aplicado igual en g, r y z | Con la misma apertura en las tres bandas, el color compara la misma región de la galaxia |
| Radios R20, R50, R80, R90 | Radios que contienen esa fracción del flujo de Petrosian | Tamaño aparente y concentración |
| Morfología | C, A, S (Conselice 2003), Gini y M20 (Lotz et al. 2004), b/a | Distribución de la luz: bulbo frente a disco, perturbaciones, formación estelar grumosa |

Para la morfología seguimos las definiciones y parámetros de `statmorph` (Rodriguez-Gomez et al. 2019), con dos diferencias: el término de ruido de A y S se calcula a partir de σ del cielo en lugar de medirse en una caja de fondo, y todas las aperturas son circulares, igual que el `PETRO_TH90` de NSA con el que comparamos.

El código de medición está en `src/caracteristicas_desi/` y se verificó con pruebas automáticas sobre galaxias sintéticas de Sérsic, cuyos radios, flujos y concentraciones se conocen analíticamente (`tests/caracteristicas_desi/`). La siguiente celda mide las 9,303 galaxias en paralelo y guarda cada bloque al terminar. Al final les une la grumosidad calculada en la sección 4.1, que se explica en la sección 6.7.

In [ ]:
t0 = time.time()
medidas, ESTAMPAS, ids_estampas = medir_cache(CACHE, RESULTADOS / "mediciones", guard,
                                              ids_validos=set(datos.mangaid), workers=os.cpu_count())
print(f"Medidas: {len(medidas)} galaxias en {time.time() - t0:.0f} s")
print("Estado:", medidas.estado.value_counts().to_dict())
fallos = medidas[medidas.estado != "ok"]
if len(fallos):
    display(fallos[["mangaid", "motivo"]].head(10))
# Grumosidad (sección 6.7), calculada en el recorrido de la sección 4.1
medidas = medidas.merge(EQUIPO[["mangaid", "grumosidad_g", "grumosidad_r", "grumosidad_gauss_r"]],
                        on="mangaid", how="left", validate="one_to_one")

### 6.2 Ejemplo de medición en una galaxia

Para revisar el procedimiento tomamos una galaxia de entrenamiento de masa intermedia. A la izquierda se muestra la imagen en color (z, r, g → rojo, verde, azul; Lupton et al. 2004) con la máscara de la galaxia (blanco), los vecinos enmascarados (rojo) y la apertura de Petrosian (amarillo, 2 R_P). Al centro aparece la imagen limpia sobre la que se mide, y a la derecha la curva de crecimiento, es decir, la fracción del flujo contenida dentro de cada radio, con R50, R90 y R_P marcados.

In [ ]:
from astropy.visualization import make_lupton_rgb

def leer_recorte(mangaid):
    # Busca el recorte de una galaxia en los bloques de la caché (solo lectura).
    for f in sorted(CACHE.glob("chunk_*.npz")):
        with np.load(f) as z:
            k = np.where(z["mangaid"].astype(str) == mangaid)[0]
            if len(k):
                return z["x"][k[0]].astype(np.float32)
    raise KeyError(mangaid)

ok_tr = datos[(datos.particion == "Train") & datos.mangaid.isin(medidas.loc[medidas.estado == "ok", "mangaid"])]
ejemplo = ok_tr.iloc[(ok_tr.log_mstar - ok_tr.log_mstar.median()).abs().argsort().iloc[0]]
m = medir_fotometria(leer_recorte(ejemplo.mangaid))
n = m.limpio.shape[-1]; c = (n - 1) / 2

fig, ax = plt.subplots(1, 3, figsize=(13, 4.2))
rgb = lambda x: make_lupton_rgb(x[2], x[1], x[0], Q=8, stretch=0.05, minimum=0)
cubo_ej = leer_recorte(ejemplo.mangaid)
ax[0].imshow(rgb(cubo_ej), origin="lower")
ax[0].contour(m.mascara_gal, [0.5], colors="w", linewidths=0.8)
ax[0].contour(m.mascara_vec, [0.5], colors="r", linewidths=0.6)
ax[0].add_patch(plt.Circle((c, c), m.r_ap_px, fill=False, color="y", lw=1))
ax[0].set_title(f"{ejemplo.mangaid} · log M* = {ejemplo.log_mstar:.2f}")
ax[1].imshow(rgb(m.limpio), origin="lower"); ax[1].set_title("Cielo restado y vecinos rellenados")
p = m.perfiles[1]; R = p.radios * FE.PIXSCALE
ax[2].plot(R, p.flujo / p.F(m.r_ap_px), "k-")
for nombre, col in (("R50", "C0"), ("R90", "C1"), ("R_P", "C2")):
    ax[2].axvline(m.valores[nombre], color=col, ls="--", label=f"{nombre} = {m.valores[nombre]:.1f}″")
ax[2].set(xlim=(0, 2.5 * m.valores["R_P"]), ylim=(0, 1.1), xlabel="radio [″]", ylabel="F(<R) / F_Petrosian",
          title="Curva de crecimiento en r"); ax[2].legend()
for a in ax[:2]: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

### 6.3 Extinción galáctica y posición en el cielo

La luz de cada galaxia atraviesa el polvo de la Vía Láctea, que la atenúa y la enrojece en distinta medida según la dirección. Como este efecto no es propio de la galaxia, lo corregimos: E(B−V) se lee del mapa de Schlegel, Finkbeiner & Davis (1998) y se multiplica por los coeficientes A/E(B−V) de Legacy Surveys DR10, que son 3.214, 2.165 y 1.211 para g, r y z (Schlafly & Finkbeiner 2011).

A partir de la posición también construimos `region_norte`. Las imágenes al norte de Dec = 32.375° provienen de BASS y MzLS, y las del sur de DECam (Dey et al. 2019). Como estos telescopios tienen filtros, profundidad y resolución distintos, la variable nos permite detectar diferencias sistemáticas entre regiones (sección 11).

In [ ]:
SFD_DIR = Path(env("AV2_SFD_DIR", str(RESULTADOS / "externos" / "sfd")))
if not env("AV2_SFD_DIR"):
    guard.mkdir(SFD_DIR)
    for nombre, url in FE.SFD_URLS.items():
        guard.download(url, SFD_DIR / nombre)
datos["ebv"] = ebv_sfd(datos.ra.to_numpy(), datos.dec.to_numpy(), SFD_DIR)
datos["region_norte"] = (datos.dec >= FE.DEC_NORTE).astype(int)
print("E(B−V): mediana", round(datos.ebv.median(), 3), "· máximo", round(datos.ebv.max(), 3))
print("Galaxias en el norte (BASS/MzLS):", int(datos.region_norte.sum()), "de", len(datos))

### 6.4 Variables físicas

Con el redshift pasamos de lo **aparente** (cómo se ve la galaxia desde aquí) a lo **intrínseco** (cómo es). Esta conversión es especialmente relevante en nuestro caso: la red convolucional mostró un sesgo en masa que depende de la distancia, pues sobreestima las galaxias cercanas y subestima las lejanas. Con la imagen sola no es posible distinguir una galaxia pequeña y cercana de una grande y lejana, y el redshift resuelve esa ambigüedad.

| Variable | Construcción | Fuente |
|---|---|---|
| Magnitudes corregidas `m_b0`, colores `g_r`, `r_z`, `g_z` | m − A_b · E(B−V) | Schlafly & Finkbeiner 2011 |
| Gradientes de color `delta_g_r`, `delta_r_z` | Color dentro de R50 menos color entre R50 y R90 | — |
| Brillo superficial `mu50_b` | m + 2.5 log10(2π R50²) | Blanton et al. 2001 |
| Módulo de distancia `DM` y magnitudes absolutas `M_b` | ΛCDM plana, H0 = 73 km/s/Mpc, Ωm = 0.3 | La cosmología de Pipe3D (Sánchez et al. 2022), para que todo sea coherente con las etiquetas |
| Luminosidad `log_L_r` | 0.4 (4.65 − M_r) | M☉,r = 4.65 (Willmer 2018) |
| Tamaño físico `log_R50_kpc` y densidad `log_SigmaL` | Escala angular a la distancia de la galaxia | — |
| Masa fotométrica `logM_ebrova` | 0.673 M_g − 1.108 M_r + 0.996 | Ebrová et al. 2025, calibrada para Legacy Surveys |
| Masa por color `logM_bell` | log L_r − 0.306 + 1.097 (g − r) | Bell et al. 2003 |
| Densidad de masa `log_SigmaM` | logM − log(2π R50²) | Relacionada con el cese de la formación estelar |

**Corrección K.** No la aplicamos. Esta corrección compensa que, a mayor redshift, cada filtro observa una región más azul del espectro. A la mediana de MaNGA (z ≈ 0.04) equivale a unas centésimas de magnitud en g y r, por lo que conservamos el redshift como variable para que el modelo absorba el efecto restante.

In [ ]:
df = datos.merge(medidas, on="mangaid", how="inner", validate="one_to_one")
df = agregar_variables_fisicas(df)
df["apertura_truncada"] = df.apertura_truncada.astype(float)
df = df.replace([np.inf, -np.inf], np.nan)
CONSTRUIDAS = [v for v in DICCIONARIO.variable if v in df.columns]
# Una columna sin ningún valor en entrenamiento, o constante, no aporta información y
# SimpleImputer la eliminaría en silencio. Se quita aquí y se informa.
_tr = df.particion == "Train"
_vacias = [v for v in CONSTRUIDAS if df.loc[_tr, v].notna().sum() == 0 or df.loc[_tr, v].nunique(dropna=True) <= 1]
if _vacias:
    print("Sin valores o constantes en entrenamiento (se excluyen):", _vacias)
    CONSTRUIDAS = [v for v in CONSTRUIDAS if v not in _vacias]
print(f"{len(df)} galaxias × {len(CONSTRUIDAS)} características construidas")

### 6.5 Validación de las mediciones

Antes de usar las características verificamos que midan lo que deben. Para ello comparamos tres mediciones con catálogos independientes, solo en entrenamiento:

- **Tamaño:** nuestro R90 frente al `PETRO_TH90` de NSA, medido con imágenes SDSS.
- **Concentración:** nuestra C frente a la del catálogo auxiliar.
- **Masa:** nuestras dos masas fotométricas frente a la de Pipe3D (la etiqueta) y a la de NSA.

Resumimos la dispersión con σ_NMAD = 1.4826 × mediana(|Δ − mediana(Δ)|), que es poco sensible a valores extremos.

In [ ]:
tr = df.particion == "Train"
def nmad(x):
    x = x[np.isfinite(x)]; return 1.4826 * np.median(np.abs(x - np.median(x)))

pares = [("R90", "PETRO_TH90", "R90 [″]"), ("C", "C_catalogo", "C"),
         ("logM_ebrova", "log_mstar", "log M* [dex]"), ("logM_bell", "log_mstar", "log M* [dex]")]
fig, ax = plt.subplots(1, 4, figsize=(15, 3.6))
filas = []
for a, (x, y, et) in zip(ax, pares):
    d = df.loc[tr, [x, y]].dropna()
    dif = d[x] - d[y] if "logM" in x or x == "C" else np.log10(d[x] / d[y])
    filas.append({"medición": x, "referencia": y, "N": len(d), "Spearman ρ": stats.spearmanr(d[x], d[y])[0],
                  "desplazamiento mediano": np.median(dif), "σ_NMAD": nmad(dif.to_numpy())})
    a.hexbin(d[y], d[x], gridsize=50, mincnt=1, bins="log", cmap="viridis",
             xscale="log" if x == "R90" else "linear", yscale="log" if x == "R90" else "linear")
    lim = [np.nanpercentile(d.values, 0.5), np.nanpercentile(d.values, 99.5)]
    a.plot(lim, lim, "r--", lw=1); a.set(xlabel=f"{y}", ylabel=f"{x}", title=et)
plt.tight_layout(); plt.show()
validacion = pd.DataFrame(filas)
nsa = df.loc[tr, ["logM_ebrova", "LogMassNSA"]].dropna()
validacion.loc[len(validacion)] = ["logM_ebrova", "LogMassNSA", len(nsa), stats.spearmanr(nsa.iloc[:, 0], nsa.iloc[:, 1])[0],
                                   np.median(nsa.iloc[:, 0] - nsa.iloc[:, 1]), nmad((nsa.iloc[:, 0] - nsa.iloc[:, 1]).to_numpy())]
display(validacion)

**Por qué lo hacemos.** Antes de usar las características hay que comprobar que miden lo que dicen medir. Un error en el código de medición se propagaría a todo lo demás, así que comparamos nuestras mediciones con catálogos independientes que midieron lo mismo con otros datos.

**Qué observamos.** Las mediciones coinciden bien:

- **Tamaño:** nuestro R90 y el de NSA tienen una correlación de 0.92; el nuestro es en promedio un 5 % menor.
- **Concentración:** la correlación con el catálogo auxiliar es 0.91, prácticamente sin desplazamiento.
- **Masa:** las dos masas fotométricas siguen a la de Pipe3D con una correlación de 0.98 y una dispersión de unos 0.13 dex. La de Ebrová et al. queda 0.08 dex por debajo y la de Bell et al. 0.05 dex por encima.

En las gráficas hay unos pocos puntos muy alejados, que corresponden a galaxias con imágenes defectuosas o superpuestas con otra galaxia; son pocos y no cambian la conclusión. La masa de NSA queda 0.36 dex por debajo de la nuestra, sobre todo porque cada catálogo supone una función inicial de masa (IMF) distinta: Ebrová et al. y Pipe3D usan la de Salpeter y NSA la de Chabrier, que da masas unos 0.25 dex menores (Ebrová et al. 2025). Concluimos que el código mide correctamente.

### 6.6 Diccionario de características

Documentamos cada característica con su definición, unidades, motivo físico y referencia, y guardamos el diccionario en `resultados_fe/diccionario_caracteristicas.csv`.

Agrupamos las características en cuatro tipos, que usamos para formar los conjuntos de la sección 14:

| Grupo | Qué contiene |
|---|---|
| **aparente** | Lo que se mide en la imagen sin conocer la distancia |
| **intrínseca** | Lo que requiere el redshift |
| **estructura** | Tamaño y distribución de la luz |
| **contexto** | Condiciones de observación, que pueden introducir sesgos |

El ángulo de posición se mide, pero **no** lo usamos como característica, porque la orientación de una galaxia en el cielo no tiene relación con sus propiedades físicas.

In [ ]:
display(DICCIONARIO.groupby("grupo").variable.count().rename("características"))
display(DICCIONARIO)
guard.write(RESULTADOS / "diccionario_caracteristicas.csv", lambda t: DICCIONARIO.to_csv(t, index=False), allow_replace=True)

### 6.7 Canales de grumosidad y de masa (aporte del Preprocesamiento)

El cuaderno de Preprocesamiento propuso construir, además de la imagen, **canales** que resalten propiedades concretas:

- **Canal de grumosidad:** la imagen menos una versión suavizada de sí misma. Lo que queda son las estructuras pequeñas y brillantes, como los cúmulos de estrellas jóvenes en los brazos espirales. Por eso debería relacionarse con la formación estelar. Se mide en las bandas azules (g, r), donde esas estrellas brillan más.
- **Canal de masa:** las bandas más rojas (i y z en su galaxia de prueba), dominadas por estrellas viejas, que son la mayor parte de la masa. Nuestros cubos no tienen banda i, así que este canal es la banda z dentro de la máscara. Su información ya está en las características de la sección 6 (`m_z0`, `M_z`, `r_z`), así que no construimos otra variable con él.

Para usar el canal de grumosidad en el modelo tabular lo resumimos en un número por galaxia y banda:

> **grumosidad_b** = [ Σ max(I − I_suave, 0) − ruido ] / Σ I, sumando sobre la galaxia sin su núcleo (r ≥ 16 px).

Al probarlo encontramos un problema con el suavizado original, un filtro gaussiano de σ = 8 px. En una galaxia el brillo disminuye hacia afuera, y el promedio gaussiano queda por encima del valor real en las afueras. Entonces la diferencia se vuelve negativa y la medida termina describiendo la forma del perfil y no los grumos. Lo cambiamos por un **filtro de mediana** de 17 px, de la misma escala, que no tiene ese sesgo y también elimina los grumos pequeños. Además restamos lo que aporta el ruido, medido igual en el cielo, como hace la suavidad S de Conselice (2003). La tabla siguiente compara las dos versiones en entrenamiento: si la grumosidad mide formación estelar, debe aumentar con la sSFR y disminuir con el D4000.

Las dos variables nuevas (`grumosidad_g` y `grumosidad_r`) se agregaron al diccionario y pasan por las mismas transformaciones, escalamiento y selección que las demás.

In [ ]:
tr6 = df.particion == "Train"
d6 = df.loc[tr6].assign(log_sSFR=lambda t: t.log_sfr - t.log_mstar)
cols = {"grumosidad_g": "mediana, g (adaptada)", "grumosidad_r": "mediana, r (adaptada)",
        "grumosidad_gauss_r": "gaussiana, r (original)"}
refs = ["log_sSFR", "d4000", "S", "A", "C", "gini", "log_mstar"]
display(d6[list(cols) + refs].corr(method="spearman").loc[list(cols), refs].rename(index=cols).round(2)
        .rename_axis("ρ de Spearman (entrenamiento)"))
print("Valores ausentes en grumosidad_r:", int(df.grumosidad_r.isna().sum()))

fig, ax = plt.subplots(1, 4, figsize=(15, 3.4))
x0 = leer_recorte(ejemplo.mangaid)
med_sig = [cielo(x0[b]) for b in range(3)]
s0 = x0 - np.array([m_ for m_, _ in med_sig], np.float32)[:, None, None]
gal0, _, _ = segmentar(s0[1])
fuera_nucleo = gal0 & (mapa_radial(s0.shape[-1]) >= IE.GRUM_NUCLEO_PX)
alta = np.clip(s0[0] - ndimage.median_filter(s0[0], size=IE.GRUM_MEDIANA_PX, mode="nearest"), 0, None) * fuera_nucleo
asinh = lambda a, b: np.arcsinh(a / (3 * med_sig[b][1]))
ax[0].imshow(asinh(s0[0], 0), origin="lower", cmap="gray"); ax[0].set_title(f"{ejemplo.mangaid} · banda g")
ax[1].imshow(asinh(alta, 0), origin="lower", cmap="magma"); ax[1].set_title("canal de grumosidad (g, sin el núcleo)")
ax[2].imshow(asinh(s0[2], 2) * gal0, origin="lower", cmap="gray"); ax[2].set_title("canal de masa (z, en la máscara)")
for a in ax[:3]:
    a.set_xticks([]); a.set_yticks([])
ax[3].scatter(d6.log_sSFR, d6.grumosidad_g, s=2, alpha=0.3)
ax[3].set(xlabel="log sSFR [1/año]", ylabel="grumosidad_g", ylim=np.nanpercentile(d6.grumosidad_g, [0.5, 99.5]),
          title="Grumosidad frente a formación estelar")
plt.tight_layout(); plt.show()

**Por qué lo hacemos.** En la ejecución anterior de este cuaderno, la suavidad S de la sección 6 quedó fuera en la selección, porque unas pocas imágenes defectuosas le dieron valores extremos (sección 12). La grumosidad describe lo mismo, la luz en estructuras pequeñas, pero con una escala fija y un filtro de mediana, que es menos sensible a esos casos. Así la información sobre formación estelar tiene una segunda oportunidad de entrar al modelo.

**Cómo leer el resultado.** En la prueba con 300 galaxias, la versión gaussiana original se correlacionaba al revés de lo esperado (positiva con el D4000 y negativa con la sSFR), lo que confirma que medía la forma del perfil. La versión con mediana tiene el signo esperado: más grumosidad en las galaxias que forman estrellas y en las de población joven. La tabla muestra si se mantiene en toda la muestra de entrenamiento. Si `grumosidad_g` y `grumosidad_r` resultan casi iguales entre sí, el filtro de correlación de la sección 12 conservará solo una.

## 7. Calidad de los datos: valores ausentes

Algunas mediciones no se pueden calcular en todas las galaxias:

- **R_P no converge** en las galaxias más grandes que el recorte.
- Una galaxia **no tiene redshift**, por lo que tampoco tiene variables intrínsecas.
- En galaxias muy débiles, el **flujo de alguna banda puede ser nulo o negativo**, y entonces la magnitud queda indefinida.

**Decisión.** No eliminamos galaxias, para trabajar con la misma muestra que la red. Imputamos cada valor ausente con la **mediana de entrenamiento** de la variable, que es poco sensible a las colas, y conservamos `apertura_truncada` como indicador de que la medición es menos confiable. El imputador se ajusta solo con entrenamiento y forma parte del pipeline (sección 10.3).

In [ ]:
ausentes = df.loc[df.particion != "Test", CONSTRUIDAS].isna().mean().mul(100).sort_values(ascending=False)
display(ausentes[ausentes > 0].round(2).rename("% ausente (Train + Validation)").to_frame())
print("R_P no converge:", int((~df.petro_converge.fillna(False).astype(bool)).sum()),
      "· apertura truncada:", int(df.apertura_truncada.sum()), "· fallos de medición:", int((df.estado != "ok").sum()))

## 8. Codificación y discretización

Desde aquí trabajamos con entrenamiento, para ajustar, y con validación, para medir.

### 8.1 Codificación de variables categóricas

| Variable | Categorías | Técnica | Por qué |
|---|---|---|---|
| `region_norte` | Norte (BASS/MzLS), sur (DECam) | One-hot binaria: una columna 0/1 | Son dos categorías sin orden; una sola columna es suficiente y evita la colinealidad perfecta (la *trampa de las variables ficticias*) |
| `T_11` | Tipo morfológico en la escala T de de Vaucouleurs (−5 elíptica … 10 irregular) | **Ordinal** frente a **one-hot** de clases agrupadas | El tipo morfológico tiene un orden físico (de esferoides a discos cada vez más dominantes y con más formación estelar). Comparamos si ese orden es suficiente o si conviene tratar cada clase por separado |

`T_11` proviene del catálogo auxiliar, cuya clasificación se hizo con imágenes de SDSS y no con las nuestras. Por eso no la incluimos en el conjunto principal y la evaluamos aparte en la sección 14, para medir cuánto aporta un dato externo. A las 6 galaxias sin `T_11` les asignamos la moda de entrenamiento.

In [ ]:
from sklearn.preprocessing import OneHotEncoder, KBinsDiscretizer

M = df[df.particion.isin(["Train", "Validation"])].reset_index(drop=True)
tr, va = (M.particion == "Train").to_numpy(), (M.particion == "Validation").to_numpy()
Y = {t: M[t].to_numpy() for t in OBJETIVOS}
print("Entrenamiento:", tr.sum(), "· validación:", va.sum())

# T_11: ordinal (el propio valor T) y one-hot de clases agrupadas.
moda_t = M.loc[tr, "T_11"].mode()[0]
t11 = M.T_11.fillna(moda_t)
clases_t = pd.cut(t11, [-10, -3.5, 0.5, 3.5, 7.5, 12], labels=["E", "S0", "Sa-Sb", "Sbc-Sd", "Sdm-Irr"])
T11_ORD = t11.to_frame("T_11")
clases_t = clases_t.astype(str).to_frame("T")
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore").fit(clases_t[tr])
T11_OH = pd.DataFrame(ohe.transform(clases_t), columns=ohe.get_feature_names_out())
display(pd.crosstab(clases_t["T"][tr], M.loc[tr, "region_norte"].map({0: "sur", 1: "norte"}), margins=True))

### 8.2 Discretización

Al discretizar una variable continua se pierde información, por lo que **no** discretizamos ninguna característica de entrada del modelo. La discretización solo la usamos en tres casos:

1. **Intervalos de redshift** (5 cuantiles de entrenamiento), para analizar si los errores dependen de la distancia.
2. **Clases físicas de las etiquetas**, necesarias para las pruebas de chi-cuadrado y ANOVA de clasificación de la sección 12:
   - formadora frente a apagada, con log sSFR = log SFR − log M\* < −10.8 (el corte de Katachi);
   - población joven frente a vieja, con D4000 = 1.5.
3. **Características en 10 intervalos de cuantil**, solo para chi-cuadrado, que requiere conteos por categoría.

La siguiente tabla corresponde al primer caso: el sesgo de nuestra masa fotométrica en cada intervalo de redshift.

In [ ]:
kz = KBinsDiscretizer(n_bins=5, encode="ordinal", strategy="quantile").fit(M.loc[tr, ["z"]].fillna(M.z.median()))
M["z_bin"] = kz.transform(M[["z"]].fillna(M.z.median())).astype(int)
bordes = kz.bin_edges_[0]
tabla_z = (M[tr].assign(sesgo_ebrova=M.logM_ebrova - M.log_mstar)
           .groupby("z_bin").agg(N=("z", "size"), z_min=("z", "min"), z_max=("z", "max"),
                                 log_mstar_medio=("log_mstar", "mean"), sesgo_ebrova=("sesgo_ebrova", "median")))
display(tabla_z)

# Clases físicas (solo para pruebas de filtro; nunca son características)
M["apagada"] = ((M.log_sfr - M.log_mstar) < FE.CORTE_SSFR).astype(int)
M["vieja"] = (M.d4000 > FE.CORTE_D4000).astype(int)
print("Fracción apagadas:", round(M.loc[tr, "apagada"].mean(), 3), "· fracción viejas:", round(M.loc[tr, "vieja"].mean(), 3))

**Por qué lo hacemos.** Los intervalos de redshift sirven para revisar si una estimación se equivoca más con las galaxias cercanas o con las lejanas, y las clases físicas (apagada/formadora y vieja/joven) son necesarias para las pruebas de chi-cuadrado y ANOVA de la sección 12.

**Qué observamos.** El error de la masa fotométrica es casi el mismo, alrededor de −0.11 dex, en todos los intervalos salvo el más lejano. La fórmula tiene un desplazamiento fijo, que un modelo corrige fácilmente, y no un error que dependa de la distancia. La tabla también muestra que las galaxias lejanas son en promedio más masivas (9.9 en el intervalo más cercano frente a 11.4 en el más lejano), porque a mayor distancia solo se observan las más brillantes; por eso el redshift por sí solo ya predice parte de la masa (sección 14). Las clases quedan equilibradas, con 53 % de galaxias apagadas y 47 % de poblaciones viejas, lo que es adecuado para las pruebas estadísticas.

## 9. Transformación de variables asimétricas

Muchas cantidades astronómicas son asimétricas: los flujos y los tamaños tienen colas largas hacia valores altos, y la asimetría A se concentra cerca de 0 con unas cuantas galaxias perturbadas muy por encima. Una variable con cola larga genera tres problemas:

- domina la media y la varianza de un escalador estándar;
- la regresión lineal se ajusta a los valores extremos;
- los métodos basados en correlación lineal, como PCA y el análisis factorial, la representan mal.

Este es el **riesgo de características sesgadas** al que se refiere el objetivo 2.4.

**Regla de transformación.** La definimos antes de ver resultados: calculamos la asimetría (*skewness*) de cada característica en entrenamiento y aplicamos lo siguiente:

| Caso | Transformación |
|---|---|
| \|asimetría\| ≤ 1 | Ninguna |
| \|asimetría\| > 1 y la variable es estrictamente positiva | Se prueban **log10** y **Box-Cox** (Box & Cox 1964) y se elige la que deja la asimetría más cerca de 0. El logaritmo es la opción natural para cantidades multiplicativas, como flujos y radios |
| \|asimetría\| > 1 y la variable puede valer 0 o ser negativa | **Yeo-Johnson** (Yeo & Johnson 2000), que extiende Box-Cox a toda la recta real |
| Variables binarias | Nunca se transforman |

El umbral de 1 es el criterio habitual para considerar alta la asimetría (Bulmer 1979). El parámetro λ se estima por máxima verosimilitud en entrenamiento y se aplica sin cambios a validación (Galli 2022, cap. 3). Lo limitamos al intervalo [−3, 3] porque, en variables casi siempre nulas con unos pocos valores altos (por ejemplo, la fracción de vecinos), la máxima verosimilitud produce λ extremos que vuelven inestable la transformación fuera de entrenamiento. La columna `lambda_limitado` indica en qué variables se aplicó el límite.

In [ ]:
X_crudo = M[CONSTRUIDAS].astype(float)
asim = TransformadorAsimetria(umbral=UMBRAL_ASIMETRIA).fit(X_crudo[tr])
rep = asim.reporte_.assign(abs_antes=lambda d: d.asimetria_antes.abs()).sort_values("abs_antes", ascending=False)
print("Transformaciones elegidas:", rep.transformacion.value_counts().to_dict())
display(rep.drop(columns="abs_antes").round(3))
X_trans = asim.transform(X_crudo)

In [ ]:
peores = [v for v in rep.index if rep.loc[v, "transformacion"] != "ninguna"][:4]
fig, ax = plt.subplots(2, len(peores), figsize=(3.3 * len(peores), 6))
for j, v in enumerate(peores):
    for i, (X, et) in enumerate(((X_crudo, "antes"), (X_trans, rep.loc[v, "transformacion"]))):
        x = X.loc[tr, v].dropna().to_numpy()
        stats.probplot(x, dist="norm", plot=ax[i, j])
        ax[i, j].set_title(f"{v} · {et}\nasimetría = {stats.skew(x):.2f}", fontsize=8)
        ax[i, j].set_xlabel(""); ax[i, j].set_ylabel("")
plt.suptitle("Gráficas Q-Q contra la normal (entrenamiento): las cuatro variables más asimétricas", y=1.01)
plt.tight_layout(); plt.show()

**Por qué lo hacemos.** Una variable con una cola muy larga hace que unos pocos valores extremos dominen la media, la regresión lineal y el PCA. Transformarla para que sea más simétrica reduce ese riesgo, que es el de características sesgadas del objetivo 2.4.

**Qué observamos.** De las 49 características, 24 no necesitaron transformación, 12 se transformaron con Box-Cox y 13 con Yeo-Johnson. En la mayoría la corrección funcionó: el ruido del cielo pasó de una asimetría de 60 a −0.3, y el tamaño R20 de 6.8 a prácticamente 0. Algunas variables siguieron siendo asimétricas, como la suavidad S, el gradiente de color g−r y los colores r−z. Las gráficas Q-Q muestran la razón: unas pocas galaxias tienen valores extremos que no son físicos, causados por imágenes con defectos como la de la sección 5.4. Una transformación corrige una cola larga, pero no unos cuantos valores aislados. Estas galaxias también afectan la selección de la sección 12.

## 10. Escalamiento

Las características tienen unidades y escalas muy distintas: las magnitudes rondan 15, los radios 10″, E(B−V) 0.03 y el número de fuentes vecinas 30. Los modelos basados en distancias, gradientes o penalizaciones (regresión regularizada, redes neuronales, PCA) dan más peso a las variables de mayor escala aunque no sean las más informativas. Con el escalamiento buscamos que **todas tengan un impacto comparable** en el modelo.

| Escalador | Transformación | Cuándo conviene |
|---|---|---|
| `StandardScaler` | (x − media) / desviación | Opción por defecto: centra en 0 con varianza 1, que es lo que suponen PCA, el análisis factorial y la inicialización de redes neuronales |
| `RobustScaler` | (x − mediana) / rango intercuartílico | Cuando quedan valores extremos, ya que la mediana y los cuartiles casi no cambian con ellos |
| `MinMaxScaler` | (x − mín) / (máx − mín) → [0, 1] | Cuando un método exige valores acotados o no negativos (umbral de varianza, chi-cuadrado) |

### 10.1 Elección del escalador

Para elegir, aplicamos cada escalador después de la transformación de la sección 9 y comparamos el RMSE en validación de una regresión lineal regularizada (Ridge), que es sensible a la escala de las variables.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.linear_model import Ridge

filas = []
for nombre, esc in (("StandardScaler", StandardScaler()), ("RobustScaler", RobustScaler()), ("MinMaxScaler", MinMaxScaler())):
    pre = Pipeline([("imputar", SimpleImputer(strategy="median")), ("escalar", esc)]).fit(X_trans[tr])
    Z = pre.transform(X_trans)
    fila = {"escalador": nombre}
    for t in OBJETIVOS:
        fila[NOMBRES[t]] = rmse(Y[t][va], Ridge(alpha=1.0).fit(Z[tr], Y[t][tr]).predict(Z[va]))
    filas.append(fila)
display(pd.DataFrame(filas).set_index("escalador").round(4))

**Por qué lo hacemos.** Hay varias formas de escalar, y la mejor depende de los datos. En lugar de elegir por costumbre, comparamos las tres con una prueba de error.

**Qué observamos.** `StandardScaler` y `RobustScaler` dan el mismo error en las tres etiquetas, y `MinMaxScaler` es ligeramente peor. Como después de la sección 9 las variables ya son casi simétricas, los dos primeros se comportan igual. MinMax queda atrás porque los pocos valores extremos fijan el mínimo y el máximo y comprimen al resto de las galaxias. Conservamos `StandardScaler`, que además es el que suponen PCA y el análisis factorial.

### 10.2 Prueba de convergencia (objetivo 2.4)

Además de equilibrar el peso de las variables, el escalamiento **acelera la convergencia** de los algoritmos que se optimizan por gradiente. Cuando las variables tienen escalas muy distintas, la función de pérdida forma un valle alargado: el gradiente no apunta hacia el mínimo y el optimizador avanza en zigzag. Lo medimos de tres formas, con la misma muestra y en tres versiones de los datos:

1. **Número de condición** de la matriz de diseño, es decir, el cociente entre su mayor y su menor valor singular. Mientras más grande, peor condicionado está el problema.
2. **Iteraciones de `SGDRegressor`** hasta converger, y su RMSE en validación.
3. **Curva de pérdida de `MLPRegressor`**, una red neuronal pequeña.

Las tres versiones son: (a) crudas, solo imputadas; (b) imputadas y estandarizadas; (c) transformadas, imputadas y estandarizadas.

Para esta prueba usamos un subconjunto fijo de 14 características sin combinaciones lineales exactas. Con todas, la matriz sería singular, porque algunas variables son sumas exactas de otras (g − z = (g − r) + (r − z); M_b = m_b0 − DM), y el número de condición sería infinito con cualquier escalamiento. Esa redundancia la tratamos en la selección de la sección 12.

In [ ]:
from sklearn.linear_model import SGDRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.exceptions import ConvergenceWarning

SUB = ["m_r0", "g_r", "r_z", "R50", "C", "A", "S", "gini", "m20", "b_a", "ebv", "z", "n_fuentes", "sigma_cielo_r"]
imp = lambda: SimpleImputer(strategy="median")
versiones = {
    "(a) crudas": Pipeline([("imputar", imp())]).fit(X_crudo.loc[tr, SUB]).transform(X_crudo[SUB]),
    "(b) estandarizadas": Pipeline([("imputar", imp()), ("escalar", StandardScaler())]).fit(X_crudo.loc[tr, SUB]).transform(X_crudo[SUB]),
    "(c) transformadas + estandarizadas": Pipeline([("imputar", imp()), ("escalar", StandardScaler())]).fit(X_trans.loc[tr, SUB]).transform(X_trans[SUB]),
}
filas, curvas = [], {}
y_tr, y_va = Y["log_mstar"][tr], Y["log_mstar"][va]
for nombre, Z in versiones.items():
    cond = np.linalg.cond(np.c_[np.ones(tr.sum()), Z[tr]])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        t0 = time.time()
        sgd = SGDRegressor(max_iter=5000, tol=1e-5, random_state=SEMILLA).fit(Z[tr], y_tr)
        t_sgd = time.time() - t0
        p = sgd.predict(Z[va])
        mlp = MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=300, random_state=SEMILLA).fit(Z[tr], y_tr)
    curvas[nombre] = mlp.loss_curve_
    filas.append({"datos": nombre, "número de condición": cond, "iteraciones SGD": sgd.n_iter_, "tiempo SGD [s]": t_sgd,
                  "RMSE SGD (validación)": rmse(y_va, p) if np.isfinite(p).all() else np.inf,
                  "RMSE MLP (validación)": rmse(y_va, mlp.predict(Z[va])), "iteraciones MLP": mlp.n_iter_})
display(pd.DataFrame(filas).set_index("datos"))

fig, ax = plt.subplots(figsize=(6, 3.5))
for nombre, cur in curvas.items():
    ax.plot(np.arange(1, len(cur) + 1), cur, label=nombre)
ax.set(yscale="log", xlabel="época", ylabel="pérdida de entrenamiento (MSE/2)", title="MLPRegressor · log M*")
ax.legend(); plt.tight_layout(); plt.show()

**Por qué lo hacemos.** El objetivo 2.4 pide acelerar la convergencia de los algoritmos. Medimos con tres indicadores si la transformación y el escalamiento realmente la mejoran, en lugar de suponerlo.

**Qué observamos.** El preprocesamiento mejora mucho la convergencia:

- El **número de condición** baja de 2,549 con los datos crudos a 9.7 al estandarizar y a 4.5 al transformar y estandarizar.
- **SGD** no converge con los datos crudos ni solo estandarizados: sus errores son enormes. Con los datos transformados y estandarizados converge en 11 iteraciones, con un RMSE de 0.145 dex.
- **La red MLP** baja su error de 0.221 a 0.157 y a 0.143 dex, con 230, 147 y 121 iteraciones. En su curva de pérdida, la versión transformada baja de forma más estable y llega al valor más bajo.

El escalamiento corrige la diferencia de escalas y la transformación los valores extremos; para entrenar de forma estable se necesitan los dos.

### 10.3 Pipeline de preparación

Las decisiones de las secciones 7, 9 y 10 quedan encadenadas en un solo `Pipeline` de scikit-learn:

**transformación por asimetría → imputación con la mediana → `StandardScaler`**

El pipeline se ajusta una sola vez con entrenamiento y se guarda en `resultados_fe/pipeline_preparacion.joblib`. De esta forma, en la siguiente fase se pueden aplicar exactamente las mismas operaciones a cualquier galaxia sin volver a ajustar nada, lo que corresponde a la tarea de CRISP-ML(Q) de construir pipelines de transformación reproducibles.

In [ ]:
import joblib
PREP = Pipeline([("asimetria", TransformadorAsimetria(umbral=UMBRAL_ASIMETRIA)),
                 ("imputar", SimpleImputer(strategy="median")),
                 ("escalar", StandardScaler())]).fit(X_crudo[tr])
Z = PREP.transform(X_crudo)
assert Z.shape[1] == len(CONSTRUIDAS), f"El pipeline devolvió {Z.shape[1]} columnas en lugar de {len(CONSTRUIDAS)}"
Z = pd.DataFrame(Z, columns=CONSTRUIDAS)
guard.write(RESULTADOS / "pipeline_preparacion.joblib", lambda t: joblib.dump(PREP, t), allow_replace=True)
print("Media en entrenamiento (debe ser ≈ 0):", round(float(np.abs(Z[tr].mean()).max()), 6),
      "· desviación (≈ 1):", round(float(Z[tr].std(ddof=0).mean()), 4))

## 11. Sesgos por condiciones de observación

Consideramos que una característica está **sesgada** cuando su valor depende de cómo se observó la galaxia y no solo de cómo es; un modelo que aprende de ella traslada ese sesgo a sus predicciones. Revisamos dos fuentes posibles:

1. **El telescopio.** Las galaxias del norte se observaron con BASS y MzLS, y las del sur con DECam. Sus imágenes tienen filtros, profundidad y resolución distintos, y una red entrenada con imágenes de una región no puede aplicarse directamente a la otra (Ye et al. 2025). Comparamos cada característica entre regiones con la **diferencia estandarizada de medias** (SMD: diferencia de medias dividida entre la desviación conjunta; un valor > 0.1 indica desequilibrio, según Austin 2009) y la prueba de **Kolmogorov-Smirnov**. Para no confundir el efecto del telescopio con diferencias en la población, hacemos la comparación dentro de intervalos de masa estelar.
2. **La distancia.** Una galaxia más lejana ocupa menos píxeles y la PSF suaviza más su perfil, por lo que su concentración y su Gini aparentes disminuyen aunque la galaxia sea la misma. Medimos la correlación de cada característica estructural con el redshift **a masa fija**.

In [ ]:
D = M[tr].copy()
D["bin_m"] = pd.cut(D.log_mstar, [8, 9.5, 10, 10.5, 11, 12.5])
filas = []
for v in CONSTRUIDAS:
    if v in ("region_norte", "ebv"):      # describen la posición en el cielo por definición
        continue
    smd, ks = [], []
    for _, g in D.groupby("bin_m", observed=True):
        a, b = g.loc[g.region_norte == 1, v].dropna(), g.loc[g.region_norte == 0, v].dropna()
        if len(a) > 20 and len(b) > 20:
            s = np.sqrt((a.var() + b.var()) / 2)
            smd.append((a.mean() - b.mean()) / s if s > 0 else 0)
            ks.append(stats.ks_2samp(a, b).statistic)
    filas.append({"variable": v, "SMD norte−sur (a masa fija, mediana)": np.median(smd) if smd else np.nan,
                  "KS (mediana)": np.median(ks) if ks else np.nan})
sesgo_region = pd.DataFrame(filas).set_index("variable")
sesgo_region = sesgo_region.reindex(sesgo_region.iloc[:, 0].abs().sort_values(ascending=False).index)
display(sesgo_region.head(12).round(3))
print("Variables con |SMD| > 0.1:", int((sesgo_region.iloc[:, 0].abs() > 0.1).sum()), "de", len(sesgo_region))

In [ ]:
filas = []
for v in ["C", "C_sdss", "gini", "m20", "A", "S", "R50", "mu50_r", "b_a"]:
    rs = [stats.spearmanr(g.z, g[v], nan_policy="omit")[0] for _, g in D.groupby("bin_m", observed=True) if len(g) > 50]
    filas.append({"variable": v, "ρ(z) a masa fija, mediana": np.median(rs)})
display(pd.DataFrame(filas).set_index("variable").round(3))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
g = D[(D.log_mstar > 10) & (D.log_mstar < 10.5)]
ax[0].scatter(g.z, g.C, s=3, alpha=0.4); ax[0].set(xlabel="z", ylabel="C", title="Concentración con 10 < log M* < 10.5")
for reg, et in ((0, "sur (DECam)"), (1, "norte (BASS/MzLS)")):
    ax[1].hist(D.loc[D.region_norte == reg, "sigma_cielo_g"], bins=60, histtype="step", density=True, label=et)
ax[1].set(xlabel="σ del cielo en g [nmgy/px]", title="Profundidad de la imagen por región"); ax[1].legend()
plt.tight_layout(); plt.show()

**Por qué lo hacemos.** Si una característica cambia por el telescopio o por la distancia y no por la galaxia, el modelo aprende ese efecto como si fuera real y sus predicciones heredan el sesgo. Revisarlo permite saber qué variables son confiables y si hace falta corregirlas.

**Qué observamos.**

- **Telescopio.** De 47 características, 20 difieren de forma apreciable entre las galaxias del norte y las del sur, aun comparando galaxias de masa parecida. Las diferencias más grandes están en las variables que usan la banda z, como el color r−z, porque el filtro z de los telescopios del norte y el del sur no son iguales. Las variables basadas en g−r están mucho menos afectadas.
- **Distancia.** Para galaxias de la misma masa, la concentración C y el Gini bajan un poco con la distancia, porque las galaxias lejanas se ven más borrosas. R90/R50, la asimetría y la suavidad casi no cambian, así que son las medidas de forma más estables.

En lugar de eliminar las características afectadas, le damos al modelo variables que explican el sesgo: `region_norte` y `sigma_cielo_*` describen el telescopio y la profundidad, y `z` la distancia. En la sección 14 medimos si con ellas disminuye el error.

## 12. Selección de características con métodos de filtro

Los métodos de filtro evalúan cada característica por sus propiedades estadísticas, sin entrenar el modelo final, por lo que son rápidos y no dependen de un algoritmo en particular. Los aplicamos en cascada, siempre sobre entrenamiento:

| Paso | Método | Qué elimina o mide |
|---|---|---|
| 1 | **Umbral de varianza** sobre los datos en escala [0, 1] (`MinMaxScaler`) | Variables casi constantes, que no pueden explicar diferencias entre galaxias |
| 2 | **Correlación de Spearman** con agrupamiento jerárquico, \|ρ\| > 0.95 | Redundancia: de cada grupo de variables casi idénticas (p. ej. R80 y R90, o dos magnitudes de la misma galaxia) conservamos la de mayor información mutua con las etiquetas. Spearman también detecta relaciones monótonas no lineales |
| 3 | **ANOVA F** (`f_regression`) e **información mutua** (`mutual_info_regression`) con las tres etiquetas | Relevancia lineal y no lineal para la regresión |
| 4 | **Chi-cuadrado** y **ANOVA de clasificación** (`f_classif`) contra las clases formadora/apagada y joven/vieja | Relevancia para separar poblaciones físicas. Chi-cuadrado usa las características en 10 intervalos de cuantil |
| 5 | **Curva de validación cruzada**: error frente a las k mejores del ranking combinado | Cuántas características se necesitan. Elegimos la k más pequeña cuyo error queda a menos de un error estándar del mínimo (*regla de un error estándar*; Hastie et al. 2009, §7.10) |

In [ ]:
from sklearn.feature_selection import VarianceThreshold, f_regression, mutual_info_regression, chi2, f_classif
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

Xtr_t = pd.DataFrame(SimpleImputer(strategy="median").fit(X_trans[tr]).transform(X_trans[tr]), columns=CONSTRUIDAS)

# Paso 1: umbral de varianza en escala [0, 1]
vt = VarianceThreshold(threshold=1e-3).fit(MinMaxScaler().fit_transform(Xtr_t))
var01 = pd.Series(vt.variances_, index=CONSTRUIDAS)
quedan = [v for v, k in zip(CONSTRUIDAS, vt.get_support()) if k]
print("Paso 1 · eliminadas por varianza < 0.001:", sorted(set(CONSTRUIDAS) - set(quedan)) or "ninguna")

# Información mutua media con las tres etiquetas (se usa para desempatar en el paso 2)
mi = pd.DataFrame({t: mutual_info_regression(Xtr_t[quedan], Y[t][tr], random_state=SEMILLA) for t in OBJETIVOS}, index=quedan)

# Paso 2: grupos con |ρ| > 0.95
rho = Xtr_t[quedan].corr(method="spearman").abs().fillna(0)
dist = squareform(1 - rho.to_numpy(), checks=False).clip(0)
grupos = fcluster(linkage(dist, "average"), t=0.05, criterion="distance")
redundantes, representantes = {}, []
for gid in np.unique(grupos):
    miembros = [v for v, g in zip(quedan, grupos) if g == gid]
    mejor = mi.loc[miembros].mean(1).idxmax()
    representantes.append(mejor)
    if len(miembros) > 1:
        redundantes[mejor] = [v for v in miembros if v != mejor]
print(f"Paso 2 · {len(quedan)} → {len(representantes)} variables. Grupos redundantes (se conserva la clave):")
for k, v in redundantes.items():
    print(f"   {k:16s} ⟵ {', '.join(v)}")

In [ ]:
import matplotlib.colors as mcolors
orden = [quedan[i] for i in np.argsort(grupos)]
fig, ax = plt.subplots(figsize=(11, 9.5))
im = ax.imshow(Xtr_t[orden].corr(method="spearman"), cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(orden))); ax.set_xticklabels(orden, rotation=90, fontsize=6)
ax.set_yticks(range(len(orden))); ax.set_yticklabels(orden, fontsize=6)
plt.colorbar(im, ax=ax, shrink=0.7, label="ρ de Spearman"); ax.set_title("Correlación entre características (entrenamiento), ordenadas por grupo")
plt.tight_layout(); plt.show()

In [ ]:
# Pasos 3 y 4: relevancia
R_ = representantes
puntajes = pd.DataFrame(index=R_)
for t in OBJETIVOS:
    puntajes[f"F {t}"] = f_regression(Xtr_t[R_], Y[t][tr])[0]
    puntajes[f"IM {t}"] = mi.loc[R_, t]
Xbin = KBinsDiscretizer(n_bins=10, encode="ordinal", strategy="quantile").fit_transform(Xtr_t[R_])
for clase in ("apagada", "vieja"):
    yc = M.loc[tr, clase].to_numpy()
    puntajes[f"χ² {clase}"] = chi2(Xbin, yc)[0]
    puntajes[f"F clas. {clase}"] = f_classif(Xtr_t[R_], yc)[0]
puntajes["rango medio"] = puntajes.rank(ascending=False).mean(1)
puntajes = puntajes.sort_values("rango medio")
RANKING = puntajes.index.tolist()
display(puntajes.round(3))

### 12.1 Número de características

Tomamos las k primeras características del ranking combinado y medimos el error con validación cruzada de 5 particiones **dentro de entrenamiento**, para no usar el conjunto de validación en esta decisión. Usamos `HistGradientBoostingRegressor`, que capta relaciones no lineales y no depende de la escala. Para promediar las tres etiquetas, dividimos cada RMSE entre la desviación estándar de su etiqueta.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold

def hgb():
    return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.08, random_state=SEMILLA)

ks = sorted({k for k in (3, 5, 8, 12, 16, 24, len(RANKING)) if k <= len(RANKING)})
kf = KFold(5, shuffle=True, random_state=SEMILLA)
curva = []
for k in ks:
    cols = RANKING[:k]
    por_fold = np.zeros(5)
    for t in OBJETIVOS:
        y = Y[t][tr]; s = y.std()
        for f, (a, b) in enumerate(kf.split(Xtr_t)):
            p = hgb().fit(Xtr_t.iloc[a][cols], y[a]).predict(Xtr_t.iloc[b][cols])
            por_fold[f] += rmse(y[b], p) / s / len(OBJETIVOS)
    curva.append({"k": k, "error normalizado": por_fold.mean(), "error estándar": por_fold.std(ddof=1) / np.sqrt(5)})
curva = pd.DataFrame(curva).set_index("k")
mejor = curva["error normalizado"].idxmin()
limite = curva.loc[mejor, "error normalizado"] + curva.loc[mejor, "error estándar"]
K_SEL = int(curva.index[curva["error normalizado"] <= limite].min())
SELECCION = RANKING[:K_SEL]
display(curva.round(4))
print(f"Mínimo en k = {mejor}; regla de un error estándar → k = {K_SEL}")
print("Características seleccionadas:", SELECCION)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.errorbar(curva.index, curva["error normalizado"], curva["error estándar"], marker="o", capsize=3)
ax.axhline(limite, color="gray", ls=":", label="mínimo + 1 error estándar"); ax.axvline(K_SEL, color="C3", ls="--", label=f"k elegido = {K_SEL}")
ax.set(xscale="log", xlabel="número de características (k)", ylabel="RMSE / σ(y), media de 3 etiquetas",
       title="Validación cruzada en entrenamiento"); ax.legend(); plt.tight_layout(); plt.show()
guard.write_json(RESULTADOS / "seleccion.json", {"ranking": RANKING, "k": K_SEL, "seleccion": SELECCION,
                                                  "redundantes": redundantes, "commit": COMMIT}, allow_replace=True)

**Por qué lo hacemos.** Muchas de las 49 características repiten información o casi no aportan. Con menos variables el modelo es más simple, entrena más rápido, se interpreta mejor y corre menos riesgo de aprender ruido. Los filtros eliminan las variables inútiles o repetidas y ordenan las demás por su relación con las etiquetas, y la validación cruzada decide cuántas conservar.

**Qué observamos.**

- **Umbral de varianza.** Eliminó 8 variables: A, S, los colores r−z, g−r del disco, el gradiente de color g−r, el tamaño relativo en g y la fracción de píxeles sin datos. Solo esta última es realmente casi constante. En las demás, los pocos valores extremos de la sección 9 hacen que, en la escala de 0 a 1, las demás galaxias queden apretadas en un rango muy pequeño. Es una limitación conocida de este método cuando hay valores extremos. La información de color se conserva en otras variables muy relacionadas (g−r, g−z y el color interior r−z); la asimetría y la suavidad sí se pierden.
- **Correlación.** Redujo las variables de 41 a 29 agrupando las que dicen casi lo mismo. Por ejemplo, las magnitudes absolutas, la luminosidad y las dos masas fotométricas quedaron representadas por `logM_bell`.
- **Ranking.** Los colores ocupan los primeros lugares, porque son los más útiles para la SFR, el D4000 y las dos clases. Les siguen la densidad de masa, la concentración y la masa fotométrica, que es la variable más relacionada con la masa estelar. `region_norte` queda en el último lugar: el telescopio no da información sobre las propiedades de la galaxia, que es lo deseable.
- **Número de características.** El error baja rápido hasta 8 variables y casi no cambia a partir de 12. La regla de un error estándar elige **16 de las 49** (un 67 % menos), con un error prácticamente igual al de usar todas.

## 13. Extracción de características

La selección conserva variables originales; la extracción, en cambio, las **combina** en unas pocas variables nuevas que concentran la información, a costa de parte de la interpretabilidad. Aplicamos dos técnicas sobre las características y dos sobre los píxeles:

| Técnica | Sobre | Qué produce |
|---|---|---|
| **PCA** | Las características transformadas y estandarizadas | Combinaciones lineales ortogonales, ordenadas por la varianza que explican |
| **Análisis factorial** con rotación varimax | Las mismas | Factores latentes que explican las correlaciones; la rotación los hace interpretables |
| **Eigengalaxias** (Uzeirbegovic et al. 2020) | Estampas de 32 × 32 px de cada galaxia, alineadas por su eje mayor y escaladas a ±2 R_P | Una base de imágenes; cada galaxia se resume con sus coeficientes |
| **Zoobot + PCA** (Walmsley et al. 2023), *opcional* | La imagen en color | Representaciones de una red preentrenada con millones de clasificaciones de Galaxy Zoo |

### 13.1 Análisis de componentes principales

Aplicamos PCA a las características transformadas y estandarizadas, y conservamos los componentes necesarios para explicar el 95 % de la varianza. La gráfica de cargas muestra qué variables combina cada componente.

In [ ]:
from sklearn.decomposition import PCA, FactorAnalysis

Zc = Z[CONSTRUIDAS]
pca = PCA(random_state=SEMILLA).fit(Zc[tr])
var_acum = np.cumsum(pca.explained_variance_ratio_)
N_PCA = int(np.searchsorted(var_acum, 0.95) + 1)
print(f"Componentes para explicar el 95 % de la varianza: {N_PCA} de {len(CONSTRUIDAS)}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(range(1, 21), pca.explained_variance_ratio_[:20], label="por componente")
ax[0].plot(range(1, 21), var_acum[:20], "C3o-", label="acumulada")
ax[0].axhline(0.95, color="gray", ls=":"); ax[0].set(xlabel="componente", ylabel="fracción de varianza", title="Gráfica de sedimentación")
ax[0].legend()
cargas = pd.DataFrame(pca.components_[:6].T, index=CONSTRUIDAS, columns=[f"CP{i + 1}" for i in range(6)])
top = cargas.abs().max(1).sort_values(ascending=False).index[:22]
im = ax[1].imshow(cargas.loc[top], cmap="RdBu_r", vmin=-0.5, vmax=0.5, aspect="auto")
ax[1].set_yticks(range(len(top))); ax[1].set_yticklabels(top, fontsize=7)
ax[1].set_xticks(range(6)); ax[1].set_xticklabels(cargas.columns); ax[1].set_title("Cargas de los seis primeros componentes")
plt.colorbar(im, ax=ax[1]); plt.tight_layout(); plt.show()
PCA_TR = pca.transform(Zc)[:, :N_PCA]

**Por qué lo hacemos.** PCA combina las 49 características en unos pocos componentes que concentran la información. Sirve para saber cuántas dimensiones independientes tienen realmente los datos y cuáles son.

**Qué observamos.** Se necesitan 17 componentes para explicar el 95 % de la varianza, y los dos primeros explican casi la mitad (26 % y 22 %). Las cargas permiten interpretarlos: el primero combina color y distancia, el segundo el tamaño y el tercero el brillo aparente. Los componentes 4 y 6 reflejan condiciones de observación (la banda z y el telescopio, y el ruido del cielo), lo que confirma lo visto en la sección 11.

### 13.2 Análisis factorial

El análisis factorial supone que las características observadas son combinaciones de unos pocos factores latentes más un ruido propio de cada variable. Antes de aplicarlo verificamos que sea pertinente:

- **Prueba de esfericidad de Bartlett:** comprueba que la matriz de correlaciones no sea la identidad, es decir, que las variables no sean independientes.
- **Índice KMO** (Kaiser 1974): mide qué parte de la correlación es compartida; un valor mayor que 0.6 se considera adecuado.

Elegimos el número de factores con el **análisis paralelo** de Horn (1965), que conserva los factores cuyos autovalores superan a los de datos aleatorios del mismo tamaño. Usamos el subconjunto sin redundancias del paso 2 de la selección, porque las variables casi idénticas vuelven singular la matriz de correlaciones.

In [ ]:
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity, calculate_kmo

Zf = Z[representantes]
chi2_b, p_b = calculate_bartlett_sphericity(Zf[tr])
_, kmo = calculate_kmo(Zf[tr])
print(f"Bartlett: χ² = {chi2_b:.0f}, p = {p_b:.2g} · KMO global = {kmo:.3f}")

auto = np.sort(np.linalg.eigvalsh(np.corrcoef(Zf[tr].T)))[::-1]
azar = np.mean([np.sort(np.linalg.eigvalsh(np.corrcoef(rng.standard_normal(Zf[tr].shape).T)))[::-1] for _ in range(20)], 0)
N_FA = int(np.argmax(auto < azar)) or 1
print("Factores retenidos por análisis paralelo:", N_FA)

fa = FactorAnalyzer(n_factors=N_FA, rotation="varimax" if N_FA > 1 else None, method="minres").fit(Zf[tr])
cargas_fa = pd.DataFrame(fa.loadings_, index=representantes, columns=[f"F{i + 1}" for i in range(N_FA)])
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={"width_ratios": [1, 1.3]})
ax[0].plot(range(1, len(auto) + 1), auto, "o-", label="datos"); ax[0].plot(range(1, len(azar) + 1), azar, "s--", label="aleatorio")
ax[0].axvline(N_FA + 0.5, color="C3", ls=":"); ax[0].set(xlabel="factor", ylabel="autovalor", title="Análisis paralelo"); ax[0].legend()
im = ax[1].imshow(cargas_fa, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax[1].set_yticks(range(len(representantes))); ax[1].set_yticklabels(representantes, fontsize=6)
ax[1].set_xticks(range(N_FA)); ax[1].set_xticklabels(cargas_fa.columns); ax[1].set_title("Cargas tras rotación varimax")
plt.colorbar(im, ax=ax[1]); plt.tight_layout(); plt.show()
print("Varianza explicada por factor:", np.round(fa.get_factor_variance()[1], 3))
FA_TR = fa.transform(Zf)

**Por qué lo hacemos.** El análisis factorial busca unos pocos factores ocultos que explican por qué las características varían juntas. A diferencia del PCA, produce factores más fáciles de nombrar, lo que ayuda a entender qué información contienen realmente los datos.

**Qué observamos.** Las pruebas previas indican que el método es adecuado: Bartlett rechaza que las variables sean independientes (p ≈ 0) y el KMO es 0.67, por encima de 0.6. El análisis paralelo conserva **7 factores**, que juntos explican el 62 % de la varianza:

| Factor | Qué representa | Variables con más peso |
|---|---|---|
| F1 | Tamaño aparente | Radios, brillo superficial, vecinos |
| F2 | Color o población estelar | g−r, g−z, color interior, masa fotométrica |
| F3 | Brillo aparente | Magnitudes en r y z |
| F4 | Profundidad de la imagen | Ruido del cielo en las tres bandas |
| F5 | Telescopio y banda z | Gradiente r−z, tamaño en z, región |
| F6 | Distancia y escala física | Módulo de distancia, tamaño en kpc |
| F7 | Concentración | C, R90/R50, Gini |

Dos de los siete factores (F4 y F5) describen cómo se observó la galaxia y no cómo es, lo que coincide con los sesgos de la sección 11.

### 13.3 Eigengalaxias

Las características anteriores resumen la imagen con fórmulas que elegimos nosotros. Con las eigengalaxias son los datos los que definen la representación: un PCA sobre los píxeles de miles de galaxias encuentra las *imágenes base* que, combinadas, mejor reconstruyen cada galaxia (Uzeirbegovic, Geach & Kaviraj 2020).

Para que describan **forma y color**, y no tamaño ni orientación (que ya tenemos como características), preparamos cada galaxia así:

1. se rota para que su eje mayor quede horizontal;
2. se reescala para que la estampa cubra ±2 R_P y se reduce a 32 × 32 píxeles por banda;
3. se divide entre su flujo medio en r, lo que conserva el color, y se comprime con asinh.

Cada galaxia queda descrita por 3,072 valores (3 × 32 × 32), que el PCA reduce a unas decenas de coeficientes.

In [ ]:
pos = {m: i for i, m in enumerate(ids_estampas)}
E = ESTAMPAS[[pos[m] for m in M.mangaid]].reshape(len(M), -1)
validas = np.isfinite(E).all(1)
E = np.where(validas[:, None], E, 0.0)
print("Estampas válidas:", int(validas.sum()), "de", len(M))
peig = PCA(n_components=min(60, int(tr.sum()) - 1), random_state=SEMILLA).fit(E[tr & validas])
N_EIG = int(min(np.searchsorted(np.cumsum(peig.explained_variance_ratio_), 0.90) + 1, peig.n_components_))
EIG = peig.transform(E)[:, :N_EIG]
print(f"Eigengalaxias para el 90 % de la varianza: {N_EIG}")

fig, ax = plt.subplots(2, 8, figsize=(14, 3.8))
media = peig.mean_.reshape(3, 32, 32)
for j in range(8):
    if j == 0:
        img, tit = media[1], "media (r)"
    else:
        img, tit = peig.components_[j - 1].reshape(3, 32, 32)[1], f"EG{j} ({peig.explained_variance_ratio_[j - 1]:.0%})"
    ax[0, j].imshow(img, cmap="RdBu_r" if j else "gray", origin="lower"); ax[0, j].set_title(tit, fontsize=8)
    if j:
        col = peig.components_[j - 1].reshape(3, 32, 32)
        ax[1, j].imshow(col[0] - col[2], cmap="PuOr", origin="lower"); ax[1, j].set_title("g − z", fontsize=7)
for a in ax.ravel(): a.set_xticks([]); a.set_yticks([])
ax[1, 0].axis("off")
plt.suptitle("Galaxia media y primeras eigengalaxias. Arriba: banda r. Abajo: diferencia g − z (color)", y=1.02)
plt.tight_layout(); plt.show()

**Por qué lo hacemos.** Las características anteriores las elegimos nosotros. Las eigengalaxias dejan que los datos encuentren los patrones de forma y color más comunes, por si contienen información que nuestras fórmulas no capturan.

**Qué observamos.** Las eigengalaxias muestran patrones con sentido físico. La galaxia media es una elipse alineada horizontalmente, como se esperaba tras rotarlas. La primera eigengalaxia (14 % de la varianza) distingue galaxias más o menos alargadas, la segunda galaxias con un centro más o menos concentrado, y las siguientes asimetrías y diferencias entre el centro y las afueras. Los mapas g − z muestran que varias también describen diferencias de color entre el centro y el borde. Con 60 componentes, el máximo que calculamos, no se llega al 90 % de la varianza: el ruido de cada píxel reparte la varianza en muchos componentes pequeños, así que las eigengalaxias comprimen menos que el PCA sobre características.

### 13.4 Zoobot (opcional, requiere GPU)

Zoobot (Walmsley et al. 2023) es una familia de redes entrenadas con las clasificaciones morfológicas de los voluntarios de Galaxy Zoo, que incluyen imágenes de DESI. Su penúltima capa produce una **representación** de la morfología (640 valores en ConvNeXt-Nano), que usamos como características extraídas y reducimos con PCA.

Como Zoobot se entrenó con imágenes en color de 8 bits y no con FITS, le damos la versión en color de Lupton et al. (2004) de cada galaxia, centrada y escalada a ±2 R_P.

In [ ]:
import torch
ZOO = None
if USAR_ZOOBOT == "si" or (USAR_ZOOBOT == "auto" and torch.cuda.is_available()):
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)
        import timm
        from src.caracteristicas_desi.imagenes import rgb_lupton
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        enc = timm.create_model("hf_hub:mwalmsley/zoobot-encoder-convnext_nano", pretrained=True, num_classes=0).eval().to(dev)
        rp_px = (M.set_index("mangaid").R_P / FE.PIXSCALE).to_dict()
        rep_zoo = {}
        for f in sorted(CACHE.glob("chunk_*.npz")):
            with np.load(f) as z:
                x, ids = z["x"], z["mangaid"].astype(str)
            sel = [i for i, m_ in enumerate(ids) if m_ in rp_px]
            for i0 in range(0, len(sel), 64):
                lote = sel[i0:i0 + 64]
                imgs = np.stack([rgb_lupton(x[i].astype(np.float32), rp_px[ids[i]]) for i in lote]) / 255.0
                with torch.no_grad():
                    out = enc(torch.tensor(imgs, dtype=torch.float32).permute(0, 3, 1, 2).to(dev)).cpu().numpy()
                rep_zoo.update({ids[i]: o for i, o in zip(lote, out)})
        R_zoo = np.stack([rep_zoo[m_] for m_ in M.mangaid])
        sc_zoo = StandardScaler().fit(R_zoo[tr])
        pzoo = PCA(n_components=0.90, random_state=SEMILLA).fit(sc_zoo.transform(R_zoo[tr]))
        ZOO = pzoo.transform(sc_zoo.transform(R_zoo))
        print(f"Zoobot: representación de {R_zoo.shape[1]} → {ZOO.shape[1]} componentes (90 % de la varianza)")
    except Exception as e:
        print("Zoobot no se pudo ejecutar:", type(e).__name__, str(e)[:200])
else:
    print("Sin GPU: se omite Zoobot (la comparación de la sección 14 no lo incluye).")

## 14. ¿Cuánto predicen las características?

Comparamos los conjuntos de características con dos modelos fijos, **sin ajustar hiperparámetros**, ya que lo que queremos comparar son las características y no los modelos:

- **Ridge** (regresión lineal regularizada, α = 1): es sensible a la escala y muestra cuánta información es accesible de forma lineal.
- **HistGradientBoosting**: capta relaciones no lineales e interacciones.

Ambos se ajustan con entrenamiento y se evalúan en validación. Reportamos el RMSE con su intervalo de confianza del 95 % por *bootstrap*, el número de variables, el tiempo de entrenamiento y la memoria que ocupa la matriz de entrada.

| Conjunto | Contenido |
|---|---|
| S0 | Solo el redshift (referencia mínima) |
| S1 | Fotometría aparente: magnitudes, colores, gradientes de color y brillo superficial |
| S2 | S1 + variables intrínsecas (requieren el redshift) |
| S3 | Todas las construidas (S2 + estructura + contexto) |
| S3 sin contexto | S3 sin región, profundidad ni vecinos (mide el efecto de las variables de sesgo) |
| S4 | Las seleccionadas en la sección 12 |
| S5 | Componentes principales de S3 (95 % de la varianza) |
| S6 | Factores del análisis factorial |
| S7 | Eigengalaxias |
| S8 | S4 + eigengalaxias |
| S9 | Zoobot + PCA (solo con GPU) |
| S4 + T_11 | S4 con el tipo morfológico externo, ordinal y one-hot |

In [ ]:
def estandarizar(A):
    return StandardScaler().fit(A[tr]).transform(A)

g_ap, g_in, g_es, g_cx = (grupo(n) for n in ("aparente", "intrinseca", "estructura", "contexto"))
col = lambda vs: [v for v in vs if v in CONSTRUIDAS]
CONJUNTOS = {
    "S0 · redshift": Z[["z"]].to_numpy(),
    "S1 · aparente": Z[col(g_ap)].to_numpy(),
    "S2 · + intrínseca": Z[col(g_ap + g_in)].to_numpy(),
    "S3 · todas": Z[CONSTRUIDAS].to_numpy(),
    "S3 sin contexto": Z[col(g_ap + g_in + g_es)].to_numpy(),
    f"S4 · seleccionadas ({K_SEL})": Z[SELECCION].to_numpy(),
    f"S5 · PCA ({N_PCA})": PCA_TR,
    f"S6 · factores ({N_FA})": FA_TR,
    f"S7 · eigengalaxias ({N_EIG})": estandarizar(EIG),
    "S8 · S4 + eigengalaxias": np.c_[Z[SELECCION].to_numpy(), estandarizar(EIG)],
}
if ZOO is not None:
    CONJUNTOS[f"S9 · Zoobot ({ZOO.shape[1]})"] = ZOO
CONJUNTOS["S4 + T_11 ordinal"] = np.c_[Z[SELECCION].to_numpy(), estandarizar(T11_ORD.to_numpy())]
CONJUNTOS["S4 + T_11 one-hot"] = np.c_[Z[SELECCION].to_numpy(), T11_OH.to_numpy()]

filas, PRED = [], {}
for nombre, X in CONJUNTOS.items():
    for modelo, fabrica in (("Ridge", lambda: Ridge(alpha=1.0)), ("HGB", hgb)):
        fila = {"conjunto": nombre, "modelo": modelo, "variables": X.shape[1], "memoria [MB]": X[tr].nbytes / 1e6}
        t0 = time.time()
        for t in OBJETIVOS:
            p = fabrica().fit(X[tr], Y[t][tr]).predict(X[va])
            PRED[(nombre, modelo, t)] = p
            lo, hi = rmse_ic(Y[t][va], p)
            fila[NOMBRES[t]] = rmse(Y[t][va], p)
            fila[f"IC {t}"] = f"[{lo:.3f}, {hi:.3f}]"
        fila["tiempo [s]"] = time.time() - t0
        filas.append(fila)
RES = pd.DataFrame(filas)
display(RES.set_index(["conjunto", "modelo"]).round(4))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 5.2), sharey=True)
etiquetas = list(CONJUNTOS)
yy = np.arange(len(etiquetas))
for a, t in zip(ax, OBJETIVOS):
    for k, (modelo, mk) in enumerate((("Ridge", "s"), ("HGB", "o"))):
        r = RES[RES.modelo == modelo].set_index("conjunto").loc[etiquetas]
        ic = np.array([[float(s) for s in v.strip("[]").split(",")] for v in r[f"IC {t}"]])
        a.errorbar(r[NOMBRES[t]], yy + (k - 0.5) * 0.25, xerr=np.abs(ic.T - r[NOMBRES[t]].to_numpy()),
                   fmt=mk, capsize=2, label=modelo)
    a.axvline(CNN[t], color="C3", ls="--", label="CNN (validación)")
    a.axvline(KATACHI[t], color="gray", ls=":", label="Katachi (prueba)")
    a.axvline(Y[t][va].std(), color="k", lw=0.6, label="σ de la etiqueta")
    a.set(xlabel=f"RMSE en validación · {NOMBRES[t]}")
ax[0].set_yticks(yy); ax[0].set_yticklabels(etiquetas, fontsize=8); ax[0].invert_yaxis()
ax[0].legend(fontsize=7, loc="lower right"); plt.tight_layout(); plt.show()

**Por qué lo hacemos.** Es la prueba final de todo el avance: saber si las características construidas, seleccionadas y extraídas sirven para predecir, y cómo se comparan con la red que usa la imagen completa.

**Qué observamos.** Comparamos con el mejor modelo de cada conjunto, `HistGradientBoosting`.

- **Masa estelar.** Las características con distancia predicen mucho mejor que la red: con las 16 seleccionadas el error es 0.123 dex, frente a 0.224 de la red y 0.222 de Katachi. La ventaja viene del redshift: sin él (S1, solo fotometría aparente) el error sube a 0.330, peor que la red. La red no recibe la distancia, así que el resultado muestra cuánto vale esa información, no que las características superen a la red en todo. Ridge y HGB dan errores parecidos, lo que indica que la masa depende casi linealmente del brillo real y del color, como predicen las fórmulas de la sección 6.4.
- **SFR y D4000.** La red sigue siendo mejor. El mejor conjunto, que incluye el tipo morfológico T_11, da 0.679 dex en SFR frente a 0.631 de la red, y 0.079 en D4000 frente a 0.075. Ambos superan a Katachi publicado (0.668 y 0.100). En la SFR, Ridge es mucho peor que HGB (0.88 frente a 0.69), porque las galaxias formadoras y las apagadas se comportan de forma muy distinta y la relación no es lineal.
- **Selección.** Con 16 variables en lugar de 49, el error en masa casi no cambia (0.123 frente a 0.119), y el modelo entrena en la mitad del tiempo con un tercio de la memoria.
- **Extracción.** PCA y, sobre todo, los factores pierden precisión frente a las variables seleccionadas. Las eigengalaxias solas predicen poco la masa, porque no contienen el tamaño ni el brillo, pero agregadas a las seleccionadas mejoran un poco la SFR (de 0.727 a 0.712).
- **Contexto.** Quitar las variables de telescopio y profundidad no cambia el error, así que el sesgo de la sección 11 no afecta la predicción de forma apreciable.
- **T_11.** El tipo morfológico mejora la SFR (de 0.727 a 0.679) y el D4000 (de 0.085 a 0.079). La codificación ordinal, con una columna, da el mismo resultado que la one-hot, con cinco: el orden de la escala basta.
- **Zoobot.** Sus 28 componentes, que solo describen la apariencia de la imagen en color, predicen la masa casi como el redshift solo (0.378 dex), porque las imágenes se escalan al tamaño de cada galaxia y no conservan su brillo ni su tamaño real. Para la SFR son la mejor representación basada solo en la imagen (0.828 dex, frente a 0.848 de la fotometría aparente y 0.942 de las eigengalaxias), lo que indica que la morfología que aprendió Zoobot contiene información sobre la formación estelar. Para el D4000 son peores que las eigengalaxias (0.135 frente a 0.112), probablemente porque la imagen en color de 8 bits pierde detalle del color. Aun así, quedan lejos de las características físicas: no las sustituyen, aunque podrían complementarlas.

### 14.1 Residuos frente a la distancia

La red convolucional sobreestimaba la masa de las galaxias cercanas y subestimaba la de las lejanas. Revisamos el sesgo en masa por intervalo de redshift (los de la sección 8.2) para el modelo sin variables intrínsecas (S1) y para el modelo con las características seleccionadas (S4).

In [ ]:
fila_m = []
for nombre in ("S1 · aparente", f"S4 · seleccionadas ({K_SEL})"):
    res = PRED[(nombre, "HGB", "log_mstar")] - Y["log_mstar"][va]
    s = pd.Series(res).groupby(M.loc[va, "z_bin"].to_numpy()).median()
    fila_m.append(s.rename(nombre))
sesgo_z = pd.concat(fila_m, axis=1)
sesgo_z.index = [f"{bordes[i]:.3f}–{bordes[i + 1]:.3f}" for i in sesgo_z.index]
sesgo_z.index.name = "intervalo de z"
display(sesgo_z.round(3))

**Por qué lo hacemos.** La red sobreestimaba la masa de las galaxias cercanas y subestimaba la de las lejanas. Queremos comprobar si las variables que usan la distancia corrigen ese problema.

**Qué observamos.** Sin variables de distancia (S1), el modelo comete el mismo error que la red: sobreestima la masa de las galaxias cercanas (+0.33 dex) y subestima la de las lejanas (−0.11 dex). Con las características seleccionadas (S4), el sesgo queda por debajo de 0.025 dex en todos los intervalos. Las variables intrínsecas eliminan el sesgo con la distancia, uno de los problemas principales de la red.

In [ ]:
# Productos para la siguiente fase (Model Engineering)
salida = df[["mangaid", "particion", *OBJETIVOS, *CONSTRUIDAS, "T_11"]]
guard.write(RESULTADOS / "caracteristicas_v1.parquet", lambda t: salida.to_parquet(t, index=False), allow_replace=True)
guard.write(RESULTADOS / "comparacion_validacion.csv", lambda t: RES.to_csv(t, index=False), allow_replace=True)
guard.write_json(RESULTADOS / "resumen_avance2.json", {
    "commit": COMMIT, "catalogo_sha256": SHA_CAT, "n_galaxias": int(len(df)), "n_construidas": len(CONSTRUIDAS),
    "seleccion": SELECCION, "n_pca": N_PCA, "n_factores": N_FA, "n_eigengalaxias": N_EIG}, allow_replace=True)
for p in sorted(RESULTADOS.glob("*.*")):
    print(f"  {p.name:40s} {p.stat().st_size / 1e6:7.2f} MB")

## 15. Conclusiones de la fase de preparación de los datos (CRISP-ML(Q))

En CRISP-ML(Q) (Studer et al. 2021), la fase de *Data Preparation* comprende cuatro tareas: selección de datos, limpieza, construcción de características y estandarización. Para cada tarea, la metodología pide identificar el **requisito** que se busca cumplir, el **riesgo** para la calidad del modelo y el **método de aseguramiento de calidad** que lo controla. La tabla resume cómo lo abordamos en este avance y de qué cuaderno del equipo viene cada parte.

| Tarea | Requisito | Riesgo | Control de calidad aplicado | Origen |
|---|---|---|---|---|
| **Selección de datos** | Etiquetas confiables y particiones sin contaminación | Etiquetas erróneas; fuga de información; imágenes inutilizables | Solo QCFLAG = 0; partición del proyecto; galaxias de prueba sin usar; exclusión justificada de las variables derivadas de las etiquetas (sección 4); auditoría de finitud, bandas en cero y duplicados entre particiones (sección 4.1) | Avance 2; Entrenamiento |
| **Limpieza** | Mediciones que reflejen la galaxia y no la observación | Cielo residual, vecinos, extinción, galaxias más grandes que el recorte, valores ausentes | Resta del cielo (comparada con un mapa de fondo en la sección 5.2), enmascarado y relleno de vecinos, corrección SFD, indicador `apertura_truncada`, imputación con la mediana de entrenamiento (secciones 5.2, 6 y 7) | Avance 2; Preprocesamiento |
| **Construcción** | Variables con significado físico y medidas correctamente | Mediciones con errores de código o definiciones no estándar | Definiciones publicadas; pruebas automáticas con galaxias sintéticas; validación contra NSA y el catálogo auxiliar (sección 6.5); grumosidad validada contra la sSFR y el D4000 (sección 6.7) | Avance 2; Preprocesamiento |
| **Estandarización** | Variables comparables y bien condicionadas | Características sesgadas por asimetría; dominio de las de mayor escala; convergencia lenta; sesgo por telescopio y distancia | Regla de asimetría fijada de antemano; comparación de escaladores; prueba de convergencia; SMD y KS por región; pipeline reproducible ajustado solo con entrenamiento (secciones 9–11) | Avance 2 |
| **Preparación de la imagen para la red** | Entrada de la red en una escala estable, comparable entre bandas y que conserve el brillo | Rango dinámico enorme del flujo; bandas en escalas distintas; planos y etiquetas en otra escala; normalizaciones que borran el brillo | Estiramiento asinh global por banda; comparación de 27 normalizaciones con métricas de imagen y de orden de brillo (5.3); representaciones alternativas (5.4); estandarización de etiquetas y redshift (5.5); aumentos sin relleno (5.6) | Avance 2; Preprocesamiento; Entrenamiento |
| **Selección de características** | Menor complejidad sin perder información | Redundancia; sobreajuste por exceso de variables | Umbral de varianza, correlación, F, información mutua y χ²; número de variables elegido por validación cruzada con la regla de un error estándar (sección 12) | Avance 2 |

**Respuesta a la pregunta del avance.** Con 16 características físicas y el redshift, un modelo sencillo estima la masa estelar con un error de 0.12 dex, casi la mitad que la red (0.22), y sin el sesgo con la distancia que ella tenía. Para la SFR y el D4000 la red sigue siendo mejor (0.63 frente a 0.68 dex, y 0.075 frente a 0.079), aunque las características ya superan a Katachi publicado. Las características aportan sobre todo la distancia y el color de la galaxia, y la red aprovecha mejor la información espacial de la imagen; por eso los dos enfoques se complementan.

**Lo aprendido sobre la calidad de los datos.** Unas pocas galaxias con imágenes defectuosas, por ejemplo con franjas saturadas, producen valores extremos en el ruido del cielo, la asimetría y algunos colores. Esos valores limitaron la corrección de la asimetría y llevaron al umbral de varianza a eliminar variables que sí varían. CRISP-ML(Q) es iterativo, así que en la siguiente vuelta conviene marcar esas imágenes antes de medir y aplicar el umbral de varianza sobre datos sin valores extremos.

**Qué pasa a la siguiente fase (*Model Engineering*):**

- `pipeline_preparacion.joblib`, que aplica las transformaciones sin reajustarlas.
- `caracteristicas_v1.parquet`, con todas las características medidas.
- `seleccion.json`, con el ranking y las características elegidas.
- `diccionario_caracteristicas.csv`, con la definición y referencia de cada una.

Con estos archivos podremos entrenar modelos tabulares y probar un **modelo híbrido** que combine la red convolucional con las características seleccionadas, en particular las intrínsecas, que aportan la información de distancia que la imagen por sí sola no contiene.

**Qué aprendimos como equipo.** Trabajamos por separado sobre tres partes: las características tabulares, la imagen y el protocolo de entrenamiento. Al juntarlas aparecieron dos desacuerdos y dos errores, y medirlos fue más útil que discutirlos:

- el desacuerdo sobre cómo normalizar la imagen se resolvió al ver que cada postura sirve para un uso distinto (sección 5.3);
- el canal de grumosidad necesitó un filtro distinto para medir lo que queríamos (sección 6.7);
- al correr el código en todas las galaxias aparecieron errores que con una sola galaxia no se veían: el cálculo de la saturación y la base de la entropía (sección 5.3).

**Paso a *Model Engineering*.** El cuaderno de Entrenamiento ya tiene el diseño de los experimentos de la siguiente fase. Cada variante cambia una sola cosa respecto de la anterior, para poder atribuirle su efecto:

| Variante | Qué cambia | Pregunta |
|---|---|---|
| A | Tres ResNet50 independientes, RGB, MSE | Nueva referencia, sin la cadena de Katachi |
| B | Un extractor compartido con tres salidas | ¿Ayuda compartir características entre las tres propiedades? |
| C | B + redshift después del extractor | ¿Cuánto aporta la distancia? La sección 14 sugiere que mucho para la masa |
| D | C con las bandas g, r, z transformadas en lugar de RGB | ¿Qué representación de la imagen es mejor? (sección 5.4) |
| E | D con pérdida Huber | ¿Ayuda una pérdida robusta con las galaxias atípicas? |

Las variantes se eligen con validación y con tres semillas; el grupo de prueba se abre una sola vez, al final. Además del modelo híbrido mencionado arriba, en esa fase se probará agregar a la red las características seleccionadas en la sección 12.


## 16. Referencias

**Metodología**

- Austin, P. C. (2009). Balance diagnostics for comparing the distribution of baseline covariates between treatment groups in propensity-score matched samples. *Statistics in Medicine*, 28, 3083–3107.
- Box, G. E. P. & Cox, D. R. (1964). An analysis of transformations. *Journal of the Royal Statistical Society B*, 26, 211–252.
- Bulmer, M. G. (1979). *Principles of Statistics*. Dover.
- Caselles, V., Kimmel, R. & Sapiro, G. (1997). Geodesic active contours. *International Journal of Computer Vision*, 22, 61–79.
- Deng, J., Dong, W., Socher, R., et al. (2009). ImageNet: A large-scale hierarchical image database. *Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition*, 248–255.
- Galli, S. (2022). *Python Feature Engineering Cookbook* (2.ª ed.). Packt.
- Hastie, T., Tibshirani, R. & Friedman, J. (2009). *The Elements of Statistical Learning* (2.ª ed.). Springer.
- He, K., Zhang, X., Ren, S. & Sun, J. (2016). Deep residual learning for image recognition. *Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition*, 770–778.
- Horn, J. L. (1965). A rationale and test for the number of factors in factor analysis. *Psychometrika*, 30, 179–185.
- Huber, P. J. (1964). Robust estimation of a location parameter. *The Annals of Mathematical Statistics*, 35, 73–101.
- Kaiser, H. F. (1974). An index of factorial simplicity. *Psychometrika*, 39, 31–36.
- Kendall, A., Gal, Y. & Cipolla, R. (2018). Multi-task learning using uncertainty to weigh losses for scene geometry and semantics. *Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition*, 7482–7491.
- Márquez-Neila, P., Baumela, L. & Alvarez, L. (2014). A morphological approach to curvature-based evolution of curves and surfaces. *IEEE Transactions on Pattern Analysis and Machine Intelligence*, 36, 2–17.
- Shannon, C. E. (1948). A mathematical theory of communication. *Bell System Technical Journal*, 27, 379–423.
- Studer, S., Bui, T. B., Drescher, C., et al. (2021). Towards CRISP-ML(Q): A machine learning process model with quality assurance methodology. *Machine Learning and Knowledge Extraction*, 3, 392–413.
- Yeo, I.-K. & Johnson, R. A. (2000). A new family of power transformations to improve normality or symmetry. *Biometrika*, 87, 954–959.

**Astronomía**

- Alfonzo, J. P., Iyer, K. G., Akiyama, M., et al. (2024). Decoding the imprints of past star formation on present-day morphology in galaxies with interpretable CNNs. *The Astrophysical Journal*, 967, 152.
- Bell, E. F., McIntosh, D. H., Katz, N. & Weinberg, M. D. (2003). The optical and near-infrared properties of galaxies. I. *The Astrophysical Journal Supplement Series*, 149, 289.
- Bertin, E. & Arnouts, S. (1996). SExtractor: Software for source extraction. *Astronomy and Astrophysics Supplement Series*, 117, 393–404.
- Blanton, M. R., Dalcanton, J., Eisenstein, D., et al. (2001). The luminosity function of galaxies in SDSS commissioning data. *The Astronomical Journal*, 121, 2358.
- Conselice, C. J. (2003). The relationship between stellar light distributions of galaxies and their formation histories. *The Astrophysical Journal Supplement Series*, 147, 1.
- Dey, A., Schlegel, D. J., Lang, D., et al. (2019). Overview of the DESI Legacy Imaging Surveys. *The Astronomical Journal*, 157, 168.
- Ebrová, I., Bílek, M. & Eliášek, J. (2025). Photometric stellar masses for galaxies in DESI Legacy Imaging Surveys. *Astronomy & Astrophysics*, 704, A232.
- Lotz, J. M., Primack, J. & Madau, P. (2004). A new nonparametric approach to galaxy morphological classification. *The Astronomical Journal*, 128, 163.
- Lupton, R. H., Gunn, J. E. & Szalay, A. S. (1999). A modified magnitude system that produces well-behaved magnitudes, colors, and errors even for low signal-to-noise ratio measurements. *The Astronomical Journal*, 118, 1406.
- Lupton, R., Blanton, M. R., Fekete, G., et al. (2004). Preparing red-green-blue images from CCD data. *Publications of the Astronomical Society of the Pacific*, 116, 133.
- Rodriguez-Gomez, V., Snyder, G. F., Lotz, J. M., et al. (2019). The optical morphologies of galaxies in the IllustrisTNG simulation. *Monthly Notices of the Royal Astronomical Society*, 483, 4140.
- Sánchez, S. F., Barrera-Ballesteros, J. K., Lacerda, E., et al. (2022). SDSS-IV MaNGA: pyPipe3D analysis release for 10,000 galaxies. *The Astrophysical Journal Supplement Series*, 262, 36.
- Schlafly, E. F. & Finkbeiner, D. P. (2011). Measuring reddening with Sloan Digital Sky Survey stellar spectra and recalibrating SFD. *The Astrophysical Journal*, 737, 103.
- Schlegel, D. J., Finkbeiner, D. P. & Davis, M. (1998). Maps of dust infrared emission for use in estimation of reddening and cosmic microwave background radiation foregrounds. *The Astrophysical Journal*, 500, 525.
- Strateva, I., Ivezić, Ž., Knapp, G. R., et al. (2001). Color separation of galaxy types in the Sloan Digital Sky Survey imaging data. *The Astronomical Journal*, 122, 1861.
- Uzeirbegovic, E., Geach, J. E. & Kaviraj, S. (2020). Eigengalaxies: describing galaxy morphology using principal components in image space. *Monthly Notices of the Royal Astronomical Society*, 498, 4021.
- Walmsley, M., Géron, T., Kruk, S., et al. (2023). Galaxy Zoo DESI: Detailed morphology measurements for 8.7M galaxies in the DESI Legacy Imaging Surveys. *Monthly Notices of the Royal Astronomical Society*, 526, 4768.
- Willmer, C. N. A. (2018). The absolute magnitude of the Sun in several filters. *The Astrophysical Journal Supplement Series*, 236, 47.
- Ye, R., Shen, S., de Souza, R. S., et al. (2025). From Galaxy Zoo DECaLS to BASS/MzLS: detailed galaxy morphology classification with unsupervised domain adaption. *Monthly Notices of the Royal Astronomical Society*, 537, 640.

**Software**

- Astropy Collaboration, Price-Whelan, A. M., Lim, P. L., et al. (2022). The Astropy Project: sustaining and growing a community-oriented open-source project and the latest major release (v5.0) of the core package. *The Astrophysical Journal*, 935, 167.
- Bradley, L., Sipőcz, B., Robitaille, T., et al. *photutils: Astropy package for source detection and photometry*. Zenodo. https://doi.org/10.5281/zenodo.596036
- Pedregosa, F., Varoquaux, G., Gramfort, A., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, 12, 2825–2830.
- van der Walt, S., Schönberger, J. L., Nunez-Iglesias, J., et al. (2014). scikit-image: image processing in Python. *PeerJ*, 2, e453.
- PyTorch. *Transfer learning for computer vision tutorial* y documentación de `torchvision.models.resnet50` (pesos ImageNet1K V1). https://docs.pytorch.org/
